# NYC Yellow Taxi 2026: Machine Learning with Spark MLlib

This notebook extends the Bronze → Silver → Gold pipeline with three machine-learning projects built on the cleaned Silver table (`taxi.taxi_silver.yellow_taxi_trips_cleaned`, about 12.25M trips, January–May 2026).

| Project | Task | Algorithms | Evaluation |
|---|---|---|---|
| A | Predict `fare_amount` (regression) | Linear regression, random forest, gradient-boosted trees (tuned, with a zone-pair feature) | RMSE, MAE and R² against "predict the average" and zone-pair baselines; cross-validation; saved models and MLflow runs |
| B | Predict whether a card tip is at least 20% (classification) | Logistic regression (plain, class-weighted and tuned), random forest, gradient-boosted trees (tuned) | AUC, AUC-PR, F1, decision thresholds, calibration; cross-validation by month; time split vs random split; saved model and MLflow runs |
| C | Group pickup zones by behaviour (clustering) | KMeans | Silhouette score and elbow plot; seed stability (adjusted Rand index); saved model, cluster tables and MLflow runs |

**Prerequisites:** the Silver table above, and an existing schema `taxi.taxi_ml` (this notebook does not create the schema, but it creates the two volumes it needs).

**Contents:** Setup → Project A (models, tuning, saving and tracking, zone-pair feature, results and conclusions) → Project B (classification, tuning and validation, saving and tracking) → Project C (clustering, stability, saving and tracking) → Summary

## Setup: imports

In [0]:
# Standard library
import math
import os
import json
import pandas as pd
import datetime
import sys
import logging
import warnings

# Third-party
import matplotlib.pyplot as plt
import mlflow
import mlflow.data

# PySpark: DataFrame API
from pyspark.sql import Window, functions as F

# PySpark MLlib: pipelines and feature engineering
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    Bucketizer,
    OneHotEncoder,
    SQLTransformer,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.functions import vector_to_array

# PySpark MLlib: models
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import GBTRegressor, LinearRegression, RandomForestRegressor

# PySpark MLlib: evaluators
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    ClusteringEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
)

# PySpark MLlib: tuning
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [0]:
# Spark ML on serverless needs Unity Catalog volumes: one for temporary files
# (CrossValidator and model save/load) and one for the saved models.
spark.sql("CREATE VOLUME IF NOT EXISTS taxi.taxi_ml.spark_temp")
spark.sql("CREATE VOLUME IF NOT EXISTS taxi.taxi_ml.ml_models")
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/taxi/taxi_ml/spark_temp/"

{"ts": "2026-11-28 08:40:36.031", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzc0NDE5MTUxMjQ4ODgzNxABIAEyJDAxYTIyMTRmLWYwNzItNzNmZC1hODQ1LWE2MzQyZDRkOWI5YzokNmIxMTM5MjktMTc1OC0zMWJkLThmMWEtOWZkMzI2M2U5MGRmSgwI8Kaq2AYQgMHE5wJQAVgBYAFozu3QjJDFow2IAQA=.", "context": {}}
{"ts": "2026-11-28 08:40:36.031", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzc0NDE5MTUxMjQ4ODgzNxABIAEyJDAxYTIyMTRmLWYwNzItNzNmZC1hODQ1LWE2MzQyZDRkOWI5YzokNmIxMTM5MjktMTc1OC0zMWJkLThmMWEtOWZkMzI2M2U5MGRmSgwI8Kaq2AYQgMHE5wJQAVgBYAFozu3QjJDFow2IAQA=.", "context": {}}
{"ts": "2026-11-28 08:40:36.031", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzc0NDE5MTUxMjQ4ODgzNxABIAEyJDAxYTIyMTRmLWYwNzItNzNmZC1hODQ1LWE2MzQyZDRkOWI5YzokNmIxMTM5MjktMTc1OC0zMWJkLThmMWEtOWZkMzI2

## Data snapshot and manifest helpers

A run is only reproducible if it is tied to the data it used. These helpers record, for any table, its **Delta version** (every write to a Delta table creates a new version), the time of the last write and the row count. `SILVER_SNAPSHOT` records the Silver table as it is when this notebook starts. Every MLflow model run logs it and lists its input tables as dataset inputs, and every saved model gets a `manifest.json` listing the data, features, split, settings, metrics and software versions that produced it.

In [0]:
_row_counts = {}

def table_snapshot(name):
    """Delta version, time of the last write and row count of a table, read right now."""
    last = spark.sql(f"DESCRIBE HISTORY {name} LIMIT 1").select("version", "timestamp").first()
    key = (name, int(last["version"]))
    if key not in _row_counts:                      # count again only if the table was rewritten
        _row_counts[key] = spark.table(name).count()
    return {"table": name, "delta_version": int(last["version"]),
            "last_write": str(last["timestamp"]), "rows": _row_counts[key]}

# The Silver table as it is when this notebook starts (the sample and Project C are built from it)
SILVER_SNAPSHOT = table_snapshot("taxi.taxi_silver.yellow_taxi_trips_cleaned")

# Tables each project's runs are tied to (the key is the run's task tag)
DATA_TABLES = {
    "fare": ["taxi.taxi_ml.fare_train", "taxi.taxi_ml.fare_test"],
    "tip": ["taxi.taxi_ml.tip_dataset"],
    "cluster": ["taxi.taxi_ml.zone_profile"],
}

def data_params(tables):
    """MLflow parameters that pin a run to the data it used."""
    out = {"silver_delta_version": SILVER_SNAPSHOT["delta_version"], "silver_rows": SILVER_SNAPSHOT["rows"]}
    for t in tables:
        snap = table_snapshot(t)
        short = t.split(".")[-1]
        out[f"{short}_delta_version"] = snap["delta_version"]
        out[f"{short}_rows"] = snap["rows"]
    return out

def write_manifest(model_dir, model_name, tables, feature_df, categorical, numeric,
                   split, hyperparameters, metrics, extra=None):
    """Write manifest.json into a saved model's folder: the data, features, split, settings,
    metrics and software versions that produced the model."""
    types = dict(feature_df.dtypes)
    manifest = {
        "model": model_name,
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "source_silver_table": SILVER_SNAPSHOT,
        "input_tables": [table_snapshot(t) for t in tables],
        "features": {"categorical": {c: types[c] for c in categorical},
                     "numeric": {c: types[c] for c in numeric}},
        "split": split,
        "hyperparameters": hyperparameters,
        "metrics": metrics,
        "software": {"python": sys.version.split()[0], "spark": spark.version, "mlflow": mlflow.__version__},
        **(extra or {}),
    }
    with open(f"{model_dir}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    return manifest

## Smoke test: does MLlib run on this compute?

Four toy points fitted with a linear regression. If this cell works, the Spark ML libraries are available and the rest of the notebook can run.

In [0]:
# Four toy points that lie roughly on y = 2x + 1
experiment = spark.createDataFrame(
    [(1.0, 3.1), (2.0, 5.2), (3.0, 6.9), (4.0, 9.1)], ["x", "y"]
)

# MLlib models expect all inputs packed into ONE vector column, conventionally named "features"
experiment_features = VectorAssembler(inputCols=["x"], outputCol="features").transform(experiment)

lr_model = LinearRegression(featuresCol="features", labelCol="y").fit(experiment_features)

# Expected (least squares by hand): slope 1.97, intercept 1.15
print("slope:", round(lr_model.coefficients[0], 3), "| intercept:", round(lr_model.intercept, 3))

slope: 1.97 | intercept: 1.15


## Pipeline anatomy on five toy trips

Before touching the real data, one tiny example shows how the pieces fit together.

| Stage | Type | What `.fit()` learns | What `.transform()` does |
|---|---|---|---|
| `StringIndexer` | Estimator → `StringIndexerModel` | the string → index mapping | replaces each string by its index |
| `OneHotEncoder` | Estimator → `OneHotEncoderModel` | the number of categories | turns an index into a 0/1 vector |
| `VectorAssembler` | Transformer | nothing | packs columns into one `features` vector |
| `LinearRegression` | Estimator → `LinearRegressionModel` | the coefficients | adds a `prediction` column |
| `Pipeline` | Estimator → `PipelineModel` | fits every stage in order | applies every fitted stage in order |

**Rule of thumb:** an *Estimator* has `.fit()` and returns a *Transformer* (a model); a *Transformer* has `.transform()`.

In [0]:
# Five invented trips: (trip_type, trip_distance in miles, fare_amount in $)
rows = [
    ("standard", 2.1, 12.5),
    ("airport", 17.3, 70.0),
    ("standard", 5.0, 21.0),
    ("airport", 19.0, 75.5),
    ("standard", 0.9, 8.0),
]
mini = spark.createDataFrame(rows, ["trip_type", "trip_distance", "fare_amount"])

# Estimator: learns the string -> index mapping when fitted
indexer = StringIndexer(inputCol="trip_type", outputCol="trip_type_idx")

# Estimator: turns the index into a 0/1 vector
encoder = OneHotEncoder(inputCols=["trip_type_idx"], outputCols=["trip_type_vec"])

# Transformer: packs the encoded vector and the numeric column into one "features" vector
assembler = VectorAssembler(inputCols=["trip_type_vec", "trip_distance"], outputCol="features")

# Estimator: learns the coefficients when fitted
lr = LinearRegression(featuresCol="features", labelCol="fare_amount")
print(lr.explainParams())       # every hyperparameter with its default and meaning

# A Pipeline is an Estimator too: fit() runs the stages in order and returns a PipelineModel
pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(mini)

# The PipelineModel is a Transformer: transform() applies every fitted stage
(model.transform(mini)
      .select("trip_type", "trip_distance", "fare_amount", "features", "prediction")
      .show(truncate=False))

aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0)
epsilon: The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber (default: 1.35)
featuresCol: features column name. (default: features, current: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label, current: fare_amount)
loss: The loss function to be optimized. Supported options: squaredError, huber. (default: squaredError)
maxBlockSizeInMB: maximum memory in MB for stacking input data into blocks. Data is stacked within partitions. If more than remaining data size in a partition then it is adjusted to the data size. Default 0.0 represents choosing optimal value, depends on specific algorithm. Must be >= 0. (default: 0.0)
maxIter: max number of

## Setup: the ML sample, features and task definitions

Models are trained on a **2% random sample** of `yellow_taxi_trips_cleaned` (about 245,000 trips). That is enough for stable estimates and keeps every experiment fast. The sample is written once to a Delta table in `taxi.taxi_ml`, so every later run reads exactly the same rows ("frozen") and results stay reproducible. Project C (clustering) is the exception: it aggregates the full Silver table.

### Step 1: create the sample (requires the `taxi.taxi_ml` schema)

In [0]:
silver = spark.table("taxi.taxi_silver.yellow_taxi_trips_cleaned")

# Only the columns the three projects need are copied into the ML schema
ml_columns = [
    # Where and when the trip started
    "PULocationID", "DOLocationID",
    "pickup_month", "pickup_day_of_week", "pickup_hour",
    # Trip characteristics
    "passenger_count", "trip_distance", "trip_duration_mins", "tolls_amount",
    "trip_type", "congestion_zone_flag", "payment_type",
    # Money: fare_amount is the Project A label; tip_pct is the source of the Project B label
    "fare_amount", "tip_amount", "tip_pct",
]

# sample() keeps each row with probability 0.02, so the result is ~2% of the rows, not exactly 2%.
# The seed makes the draw repeatable; saving it as a table freezes it for every later run.
(
    silver.select(ml_columns)
    .sample(fraction=0.02, seed=42)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")      # allows re-running after changing ml_columns
    .saveAsTable("taxi.taxi_ml.trips_sample")
)

print(f"Sample rows: {spark.table('taxi.taxi_ml.trips_sample').count():,}")

Sample rows: 244,506


### Step 2: time features

Raw clock values make poor model inputs, so `add_time_features` builds four better ones:

| Feature | Definition | Why |
|---|---|---|
| `is_weekend` | 1.0 if `pickup_day_of_week` is 1 or 7 | Silver numbers the days 1 = Sunday … 7 = Saturday, so the weekend is **1 and 7**, not 6 and 7 |
| `is_rush_hour` | 1.0 for pickup hours 7–9 and 16–19 | Commuter peaks change both speed and demand |
| `hour_sin`, `hour_cos` | sine and cosine of the hour placed on a 24-hour circle | Hour 23 and hour 0 are one hour apart but 23 apart as numbers; the pair puts them next to each other |

In [0]:
def add_time_features(df):
    """Add is_weekend, is_rush_hour and the cyclical hour encoding (hour_sin, hour_cos)."""
    return (
        df
        # Silver numbering: 1 = Sunday ... 7 = Saturday
        .withColumn("is_weekend",
                    F.when(F.col("pickup_day_of_week").isin(1, 7), 1.0).otherwise(0.0))
        # Commuter peaks 7-9 and 16-19; between() is inclusive on both ends
        .withColumn("is_rush_hour",
                    F.when(F.col("pickup_hour").between(7, 9) | F.col("pickup_hour").between(16, 19), 1.0)
                     .otherwise(0.0))
        # Put the 24 hours on a circle: angle = 2*pi*hour/24 = pi*hour/12
        .withColumn("hour_sin", F.sin(math.pi * F.col("pickup_hour") / 12))
        .withColumn("hour_cos", F.cos(math.pi * F.col("pickup_hour") / 12))
    )

### Step 3: look before you model

In [0]:
trips = spark.table("taxi.taxi_ml.trips_sample")

# 1. Nulls per column (VectorAssembler fails on nulls). tip_pct is expected to be null for non-card payments.
trips.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in trips.columns]).show(vertical=True)

# 2. Distribution of the Project A label. A long right tail (max far above the 99th percentile)
#    explains why RMSE ends up much larger than MAE later on.
trips.select("fare_amount").summary("min", "25%", "50%", "75%", "99%", "max", "mean", "stddev").show()

# 3. Zero-fare trips: Silver keeps fare_amount = 0. There are only 41 in this sample (0.02%),
#    so Project A keeps them. Project B filters fare_amount > 0 because a tip percentage needs a positive fare.
print("zero-fare trips:", trips.filter(F.col("fare_amount") == 0).count())

# 4. Distinct zones: one-hot encoding creates one column per zone (see build_feature_stages)
trips.select(
    F.countDistinct("PULocationID").alias("pickup_zones"),
    F.countDistinct("DOLocationID").alias("dropoff_zones"),
).show()

# 5. Rows per month: shows what a time-based split (January-April train, May test) leaves in each set
trips.groupBy("pickup_month").count().orderBy("pickup_month").show()

-RECORD 0---------------------
 PULocationID         | 0     
 DOLocationID         | 0     
 pickup_month         | 0     
 pickup_day_of_week   | 0     
 pickup_hour          | 0     
 passenger_count      | 0     
 trip_distance        | 0     
 trip_duration_mins   | 0     
 tolls_amount         | 0     
 trip_type            | 0     
 congestion_zone_flag | 0     
 payment_type         | 0     
 fare_amount          | 0     
 tip_amount           | 0     
 tip_pct              | 29465 

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|    min|               0.0|
|    25%|              10.7|
|    50%|              14.9|
|    75%|              24.0|
|    99%|              81.4|
|    max|             733.8|
|   mean|21.251186064963612|
| stddev| 18.18794226205096|
+-------+------------------+

zero-fare trips: 41
+------------+-------------+
|pickup_zones|dropoff_zones|
+------------+-------------+
|         243|          255|
+------------+----

### Step 4: decide the task before the features, and the allowed features

**Rule:** a feature is allowed only if it is known at the moment the prediction would be made. Anything computed from the label, or only available later, is *leakage* and makes results look better than they could ever be in practice.

#### Project A: fare regression (label: `fare_amount`)

| | A1: at pickup (realistic) | A2: at drop-off (sanity check) |
|---|---|---|
| Prediction moment | The passenger gets in | The trip has ended |
| Rows | All sample rows | All sample rows |
| Allowed features | `PULocationID`, `DOLocationID`, `trip_type`, `passenger_count`, `is_weekend`, `is_rush_hour`, `hour_sin`, `hour_cos` | Everything in A1, plus `trip_distance` and `trip_duration_mins` |
| Excluded | `trip_distance`, `trip_duration_mins`, `tolls_amount` (unknown at pickup); `tip_amount`, `tip_pct` (derived from the fare) | `tip_amount`, `tip_pct` (derived from the fare) |
| Note | The destination zone is assumed known, as in a fare estimate | The meter computes the fare from distance and time, so A2 mostly re-learns the meter: it is a ceiling, not a forecast |

#### Project B: high-tip classification (label: `tip_pct >= 20`)

- **Rows:** credit-card trips only (`payment_type = 1`), because cash tips are not recorded
- **Prediction moment:** the end of the trip, when the payment screen asks for a tip, so the fare, distance and duration are known
- **Allowed features:** `fare_amount`, `trip_distance`, `trip_duration_mins`, `tolls_amount`, `passenger_count`, `PULocationID`, `DOLocationID`, `trip_type`, `congestion_zone_flag`, `is_weekend`, `is_rush_hour`, `hour_sin`, `hour_cos`
- **Excluded (leakage):** `tip_amount` and `tip_pct`, because the label is built from them
- **Splits:** time-based (train January–April, test May), with a random split alongside for comparison (Project B, Step 8)

#### Project C: zone clustering (unsupervised)

- **Rows:** one row per pickup zone with at least 500 trips, aggregated from the **full** Silver table
- **Features:** log of the trip count, average distance, fare and duration, and the night, weekend and airport shares of trips
- **Label:** none. Features are standardised first because KMeans is distance-based
- **Leakage:** not applicable

### Step 5: reusable feature builder (Projects A and B)

`build_feature_stages` returns the preprocessing stages of a `Pipeline`, ending in a single `features` vector:

1. `StringIndexer` turns each categorical column into numbers, and `OneHotEncoder` turns those into 0/1 vectors.
2. The numeric columns are assembled and **standardised only when `scale=True`**: linear and logistic regression (and KMeans) are sensitive to feature scale, tree models are not.
3. A final `VectorAssembler` joins the encoded and numeric parts.

Because these stages live **inside the Pipeline**, the encoders and the scaler are fitted on the training data only and then applied unchanged to the test data. This keeps test-set information out of the preprocessing.

In [0]:
def build_feature_stages(categorical_cols, numeric_cols, scale=False):
    """Return the Pipeline stages that produce a 'features' vector column.

    categorical_cols : string/ID columns to index and one-hot encode
    numeric_cols     : numeric columns passed through (standardised if scale=True)
    scale            : True for linear/logistic regression, False for tree models
    """
    idx_cols = [c + "_idx" for c in categorical_cols]
    vec_cols = [c + "_vec" for c in categorical_cols]

    # handleInvalid="keep": a category never seen in training (e.g. a rare zone that only
    # appears in the test set) gets its own slot instead of crashing at prediction time.
    stages = [
        StringIndexer(inputCols=categorical_cols, outputCols=idx_cols, handleInvalid="keep"),
        OneHotEncoder(inputCols=idx_cols, outputCols=vec_cols, handleInvalid="keep"),
    ]

    if scale:
        # Standardise the numeric columns only; the 0/1 dummies are already on a comparable scale
        stages += [
            VectorAssembler(inputCols=numeric_cols, outputCol="numeric_raw"),
            StandardScaler(inputCol="numeric_raw", outputCol="numeric_scaled", withMean=True, withStd=True),
            VectorAssembler(inputCols=vec_cols + ["numeric_scaled"], outputCol="features"),
        ]
    else:
        stages += [VectorAssembler(inputCols=vec_cols + numeric_cols, outputCol="features")]

    return stages

In [0]:
def feature_names(df, vector_col="features"):
    """Return the names of the slots in a feature vector, in index order.

    Works on a DataFrame produced by a fitted Pipeline: VectorAssembler stores the
    slot names in the column metadata (ml_attr).
    """
    attrs = df.schema[vector_col].metadata["ml_attr"]["attrs"]
    flat = [attr for group in attrs.values() for attr in group]
    return [attr["name"] for attr in sorted(flat, key=lambda attr: attr["idx"])]

### Step 6: two extra transformers (demo only)

`Bucketizer` and `SQLTransformer` are MLlib stages that can sit inside a `Pipeline` like any other. They are shown for completeness and are **not used in the models below**.

- `Bucketizer` maps a numeric column to bucket numbers, here hour → day part (0 = night, 1 = morning, 2 = afternoon, 3 = evening).
- `SQLTransformer` runs a SQL `SELECT` on the incoming DataFrame (`__THIS__`). Here it computes the average speed. Speed needs the trip duration, so it is a *drop-off* feature and must stay out of any pickup model.

In [0]:
# Day parts: [0,6) night, [6,12) morning, [12,18) afternoon, [18,24] evening.
# The last bucket of a Bucketizer includes its upper edge, so hour 23 is covered.
day_part_bucketizer = Bucketizer(
    splits=[0, 6, 12, 18, 24],
    inputCol="pickup_hour",
    outputCol="day_part",
    handleInvalid="keep",       # null/invalid values get an extra bucket instead of raising an error
)

(day_part_bucketizer.transform(trips)
    .select("pickup_hour", "day_part").distinct()
    .orderBy("pickup_hour").show(24))

# SQL inside a Pipeline: __THIS__ stands for the incoming DataFrame
speed_transformer = SQLTransformer(statement="""
    SELECT *, ROUND(trip_distance / NULLIF(trip_duration_mins / 60, 0), 2) AS speed_mph
    FROM __THIS__
""")

speed_transformer.transform(trips).select("trip_distance", "trip_duration_mins", "speed_mph").show(5)

+-----------+--------+
|pickup_hour|day_part|
+-----------+--------+
|          0|     0.0|
|          1|     0.0|
|          2|     0.0|
|          3|     0.0|
|          4|     0.0|
|          5|     0.0|
|          6|     1.0|
|          7|     1.0|
|          8|     1.0|
|          9|     1.0|
|         10|     1.0|
|         11|     1.0|
|         12|     2.0|
|         13|     2.0|
|         14|     2.0|
|         15|     2.0|
|         16|     2.0|
|         17|     2.0|
|         18|     3.0|
|         19|     3.0|
|         20|     3.0|
|         21|     3.0|
|         22|     3.0|
|         23|     3.0|
+-----------+--------+

+-------------+------------------+---------+
|trip_distance|trip_duration_mins|speed_mph|
+-------------+------------------+---------+
|         2.66|             16.42|     9.72|
|         3.41|              14.7|    13.92|
|         4.91|              27.0|    10.91|
|         2.56|              8.18|    18.78|
|         3.09|             32.33|     5

## Project A: predicting the fare (regression)

**Goal:** predict `fare_amount` for a trip. Two variants answer different questions:

- **A1, at pickup:** zones, time of day and passenger count only. This is what a fare estimate could use.
- **A2, at drop-off:** A1 plus distance and duration. This is a ceiling that shows how much of the fare the meter inputs explain.

Models: linear regression, random forest and gradient-boosted trees, compared with baselines on a held-out test set.

### Step 1: split and freeze

A random 80/20 split, saved as two Delta tables so every later run uses exactly the same train and test rows.

In [0]:
# Random 80/20 split for Project A. Fares follow a fixed tariff (distance, time, surcharges),
# so month-to-month drift is not expected and a random split is acceptable here.
# Project B uses a time-based split instead; its Step 8 compares the two.
# The derived time features (is_weekend, is_rush_hour, hour_sin, hour_cos) from Step 2 are
# added BEFORE the split, so the frozen tables and Project B's tip_dataset carry them too.
trips = add_time_features(trips)
fare_train_raw, fare_test_raw = trips.randomSplit([0.8, 0.2], seed=42)

# Freeze both halves as Delta tables. randomSplit can return different rows if the
# partitioning of its input changes; a saved table cannot change.
fare_train_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.fare_train")
fare_test_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.fare_test")

fare_train = spark.table("taxi.taxi_ml.fare_train")
fare_test = spark.table("taxi.taxi_ml.fare_test")

print(f"train {fare_train.count():,} | test {fare_test.count():,}")

train 195,776 | test 48,730


### Step 2: evaluation helper and baselines

Every model is judged with three metrics on the held-out test set: **RMSE** (penalises large errors), **MAE** (average error in dollars) and **R²** (share of the variance explained).

A model is only worth keeping if it beats a simple baseline. Two are used: always predicting the average training fare (R² is about 0 by construction), and the average training fare of each pickup → drop-off zone pair, which is what a plain lookup table would give for free.

In [0]:
def regression_report(pred_df, label="fare_amount"):
    """RMSE, MAE and R² of a predictions DataFrame (needs a 'prediction' column and the label column)."""
    return {
        metric: round(
            RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName=metric).evaluate(pred_df),
            3,
        )
        for metric in ["rmse", "mae", "r2"]
    }


# Baseline 1: always predict the average fare of the TRAINING set (test labels stay unseen)
mean_fare = fare_train.agg(F.avg("fare_amount")).first()[0]
baseline_pred = fare_test.withColumn("prediction", F.lit(float(mean_fare)))

print("baseline (training average):", regression_report(baseline_pred))

baseline (training average): {'rmse': 18.316, 'mae': 12.21, 'r2': -0.0}


In [0]:
# Baseline 2: average training fare of each pickup -> drop-off zone pair
pair_means = (
    fare_train.groupBy("PULocationID", "DOLocationID")
    .agg(F.avg("fare_amount").alias("pair_mean_fare"))
)

pair_baseline_pred = (
    fare_test.join(pair_means, on=["PULocationID", "DOLocationID"], how="left")
    .withColumn("prediction", F.coalesce(F.col("pair_mean_fare"), F.lit(float(mean_fare))))
)

print("baseline (zone-pair average):", regression_report(pair_baseline_pred))

baseline (zone-pair average): {'rmse': 8.493, 'mae': 3.981, 'r2': 0.785}


### Step 3: linear regression with pickup features (A1)

A ridge-regularised linear model. The numeric columns are standardised, and the zones and trip type are one-hot encoded inside the Pipeline.

In [0]:
# Categorical: zones and trip type (one-hot encoded). Numeric: time features and passenger count.
fare_cat_cols = ["PULocationID", "DOLocationID", "trip_type"]
fare_num_pickup = ["passenger_count", "is_weekend", "is_rush_hour", "hour_sin", "hour_cos"]

# regParam=0.1 with elasticNetParam=0.0 is a pure L2 (ridge) penalty: it keeps the
# ~500 zone dummies from overfitting.
fare_lr = LinearRegression(featuresCol="features", labelCol="fare_amount",
                           regParam=0.1, elasticNetParam=0.0, maxIter=50)

model_lr_pickup = Pipeline(
    stages=build_feature_stages(fare_cat_cols, fare_num_pickup, scale=True) + [fare_lr]
).fit(fare_train)

pred_lr_pickup = model_lr_pickup.transform(fare_test)

print("linear, pickup features:", regression_report(pred_lr_pickup))

linear, pickup features: {'rmse': 10.697, 'mae': 6.692, 'r2': 0.659}


### Step 4: linear regression with drop-off features (A2)

The same model and hyperparameters, with `trip_distance` and `trip_duration_mins` added. Both are inputs of the taximeter, so a large improvement is expected. This variant measures a ceiling and is not a realistic forecast.

In [0]:
fare_num_dropoff = fare_num_pickup + ["trip_distance", "trip_duration_mins"]

model_lr_drop = Pipeline(
    stages=build_feature_stages(fare_cat_cols, fare_num_dropoff, scale=True) + [fare_lr]
).fit(fare_train)

pred_lr_drop = model_lr_drop.transform(fare_test)

print("linear, drop-off features:", regression_report(pred_lr_drop))

linear, drop-off features: {'rmse': 5.737, 'mae': 2.594, 'r2': 0.902}


### Step 5: tree models

Random forests and gradient-boosted trees can capture non-linear effects and interactions (for example *airport pickup × evening*) that a linear model cannot. The settings are deliberately small (`numTrees=18`, `maxIter=18`, `maxDepth=6`) so the notebook runs quickly on the sample.

One caution: one-hot encoding turns each zone into roughly 250 separate 0/1 columns, which is an awkward input for shallow trees, since each split can only isolate one zone. That is a plausible reason why trees do not beat the linear model at pickup.

In [0]:
# Hyperparameters are deliberately small so the notebook runs quickly on the 2% sample
fare_rf = RandomForestRegressor(featuresCol="features", labelCol="fare_amount",
                                numTrees=18, maxDepth=6, seed=42)
fare_gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount",
                        maxIter=18, maxDepth=6, stepSize=0.1, seed=42)

# Trees need no scaling (scale=False)
pickup_stages = build_feature_stages(fare_cat_cols, fare_num_pickup)
dropoff_stages = build_feature_stages(fare_cat_cols, fare_num_dropoff)

model_rf_pickup = Pipeline(stages=pickup_stages + [fare_rf]).fit(fare_train)
model_rf_drop = Pipeline(stages=dropoff_stages + [fare_rf]).fit(fare_train)
model_gbt_pickup = Pipeline(stages=pickup_stages + [fare_gbt]).fit(fare_train)
model_gbt_drop = Pipeline(stages=dropoff_stages + [fare_gbt]).fit(fare_train)

pred_rf_pickup = model_rf_pickup.transform(fare_test)
pred_rf_drop = model_rf_drop.transform(fare_test)
pred_gbt_pickup = model_gbt_pickup.transform(fare_test)
pred_gbt_drop = model_gbt_drop.transform(fare_test)

for name, pred in [("RF pickup", pred_rf_pickup), ("RF drop-off", pred_rf_drop),
                   ("GBT pickup", pred_gbt_pickup), ("GBT drop-off", pred_gbt_drop)]:
    print(f"{name:13s}", regression_report(pred))

RF pickup     {'rmse': 11.872, 'mae': 7.81, 'r2': 0.58}
RF drop-off   {'rmse': 5.559, 'mae': 1.934, 'r2': 0.908}
GBT pickup    {'rmse': 11.301, 'mae': 7.331, 'r2': 0.619}
GBT drop-off  {'rmse': 5.372, 'mae': 1.841, 'r2': 0.914}


## Project A: tuning and validation

So far every hyperparameter (`maxDepth=6`, `maxIter=18`, …) was picked by hand. This section chooses them with data, **without touching the test set**.

### Three sets, three jobs

| Set | Used for | How often |
|---|---|---|
| Training | fitting the model's parameters | every fit |
| Validation | choosing hyperparameters and design choices | many times |
| Test (`fare_test`) | the final, honest estimate of performance | **once** |

If the test set also guided the choices, its score would be optimistic: the winning settings would be the ones that happen to suit those particular trips.

### Cross-validation

`fare_train` is cut into 3 folds. Each candidate setting is trained on two folds and scored on the third, three times, and the three scores are averaged. Every training row is used for validation exactly once, so the score is more stable than a single validation split, at three times the cost.

The whole `Pipeline` sits inside the cross-validation, so the `StringIndexer` is refitted on each fold's training part only. `handleInvalid="keep"` matters here: a fold's training part may lack a rare zone that appears in its validation part.

### Step 1: a design choice for trees: zones without one-hot encoding

Step 5 suggested that one-hot encoding is an awkward input for trees: each zone becomes its own 0/1 column, and a depth-6 tree can only isolate a few of them. `build_tree_stages` keeps each categorical column as **one column of category indices**, and MLlib trees then split on groups of categories directly. One requirement: `maxBins` (default 32) must be at least the number of categories, and the zone columns have about 255.

This is a design choice, so it is compared on a **validation split carved out of the training set**, not on the test set.

In [0]:
def build_tree_stages(categorical_cols, numeric_cols):
    """Stages for tree models: each categorical column stays ONE column of category indices.

    VectorAssembler passes the category information on, so MLlib trees split on the categories
    directly. This needs maxBins >= the number of categories of every categorical column.
    """
    idx_cols = [c + "_idx" for c in categorical_cols]
    return [
        StringIndexer(inputCols=categorical_cols, outputCols=idx_cols, handleInvalid="keep"),
        VectorAssembler(inputCols=idx_cols + numeric_cols, outputCol="features"),
    ]


# Largest number of categories (+1 for the slot handleInvalid="keep" reserves for unseen values)
max_categories = max(
    fare_train.select(F.countDistinct(c)).first()[0] for c in ["PULocationID", "DOLocationID"]
) + 1
print(f"largest categorical column has {max_categories} values -> maxBins must be at least this")

# Validation split INSIDE the training set; fare_test is not used in this step
fare_fit, fare_val = fare_train.randomSplit([0.8, 0.2], seed=7)

# Same GBT settings as Step 5 (maxDepth=6, maxIter=18); only the encoding differs
for name, stages in [("one-hot", build_feature_stages(fare_cat_cols, fare_num_pickup)),
                     ("native categories", build_tree_stages(fare_cat_cols, fare_num_pickup))]:
    estimator = GBTRegressor(featuresCol="features", labelCol="fare_amount", maxBins=300,
                             maxDepth=6, maxIter=18, stepSize=0.1, seed=42)
    model = Pipeline(stages=stages + [estimator]).fit(fare_fit)
    print(f"{name:18s}", regression_report(model.transform(fare_val)))

largest categorical column has 255 values -> maxBins must be at least this
one-hot            {'rmse': 11.66, 'mae': 7.384, 'r2': 0.604}
native categories  {'rmse': 9.114, 'mae': 4.151, 'r2': 0.758}


**Reading Step 1.** On the validation split, native categories give RMSE $9.114 against $11.660 for one-hot encoding, so the encoding does matter for trees here. The tuning below uses native categories.

### Step 2: grid search with 3-fold cross-validation

`ParamGridBuilder` lists the candidate values; `CrossValidator` tries every combination.

| Hyperparameter | Candidates | Meaning |
|---|---|---|
| `maxDepth` | 6, 8, 10 | how detailed each tree can get (deeper trees can isolate individual routes) |
| `maxIter` | 20, 40 | number of boosting rounds (more rounds fit the training data more closely) |

That is 3 × 2 = 6 combinations × 3 folds = **18 fits**, plus one final refit of the best setting on the whole training set. The metric is RMSE, the same one used for all earlier comparisons.

In [0]:
# maxDepth and maxIter are left out here: the grid supplies them
fare_gbt_tune = GBTRegressor(featuresCol="features", labelCol="fare_amount",
                             maxBins=300, stepSize=0.1, seed=42)

# The whole Pipeline is tuned, so the StringIndexer is refitted inside every fold
tree_pipeline = Pipeline(stages=build_tree_stages(fare_cat_cols, fare_num_pickup) + [fare_gbt_tune])

param_grid = (
    ParamGridBuilder()
    .addGrid(fare_gbt_tune.maxDepth, [6, 8, 10])
    .addGrid(fare_gbt_tune.maxIter, [20, 40])
    .build()
)
print("combinations:", len(param_grid))

cv = CrossValidator(
    estimator=tree_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=2,      # fit two models at the same time
    seed=42,            # decides which rows go into which fold
)

cv_model = cv.fit(fare_train)       # the test set is NOT used here

combinations: 6


In [0]:
# One row per combination: (maxDepth, maxIter, mean validation RMSE over the 3 folds), best first
cv_results = sorted(
    [(params[fare_gbt_tune.maxDepth], params[fare_gbt_tune.maxIter], rmse)
     for params, rmse in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics)],
    key=lambda row: row[2],
)

for depth, iters, rmse in cv_results:
    print(f"maxDepth={depth:2d}  maxIter={iters:3d}  cross-validated RMSE = {rmse:.3f}")

maxDepth= 6  maxIter= 20  cross-validated RMSE = 8.780
maxDepth= 6  maxIter= 40  cross-validated RMSE = 8.804
maxDepth= 8  maxIter= 20  cross-validated RMSE = 9.163
maxDepth= 8  maxIter= 40  cross-validated RMSE = 9.437
maxDepth=10  maxIter= 20  cross-validated RMSE = 9.787
maxDepth=10  maxIter= 40  cross-validated RMSE = 9.996


### Step 2 (continued): extending the grid

A first search over maxDepth 6, 8, 10 and maxIter 20, 40 put its best setting (maxDepth 6, maxIter 20, CV RMSE $8.780) in the lowest corner of the grid, and every step towards deeper trees or more rounds made the score worse. That pattern means the optimum may lie beyond the corner, so the grid was extended downward.

| Hyperparameter | Candidates | Meaning |
| --- | --- | --- |
| `maxDepth` | 3, 4, 5, 6 | how detailed each tree can get (deeper trees can isolate individual routes) |
| `maxIter` | 10, 20, 30 | number of boosting rounds (more rounds fit the training data more closely) |

That is 4 × 3 = 12 combinations × 3 folds = **36 fits**, plus one final refit of the best setting on the whole training set. The metric is RMSE, the same one used for all earlier comparisons.

In [0]:
lower_param_grid = (
    ParamGridBuilder()
    .addGrid(fare_gbt_tune.maxDepth, [3, 4, 5, 6])
    .addGrid(fare_gbt_tune.maxIter, [10, 20, 30])
    .build()
)
print("combinations:", len(lower_param_grid))

cv2 = CrossValidator(
    estimator=tree_pipeline,
    estimatorParamMaps=lower_param_grid,
    evaluator=RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse"),
    numFolds=3,
    parallelism=2,      # fit two models at the same time
    seed=42,            # decides which rows go into which fold
)

cv_model2 = cv2.fit(fare_train)       # the test set is NOT used here

combinations: 12


In [0]:
# One row per combination: (maxDepth, maxIter, mean validation RMSE over the 3 folds), best first
cv_results2 = sorted(
    [(params[fare_gbt_tune.maxDepth], params[fare_gbt_tune.maxIter], rmse)
     for params, rmse in zip(cv_model2.getEstimatorParamMaps(), cv_model2.avgMetrics)],
    key=lambda row: row[2],
)

for depth, iters, rmse in cv_results2:
    print(f"maxDepth={depth:2d}  maxIter={iters:3d}  cross-validated RMSE = {rmse:.3f}")

maxDepth= 5  maxIter= 30  cross-validated RMSE = 8.615
maxDepth= 4  maxIter= 30  cross-validated RMSE = 8.632
maxDepth= 5  maxIter= 20  cross-validated RMSE = 8.681
maxDepth= 4  maxIter= 20  cross-validated RMSE = 8.746
maxDepth= 6  maxIter= 20  cross-validated RMSE = 8.780
maxDepth= 6  maxIter= 30  cross-validated RMSE = 8.795
maxDepth= 6  maxIter= 10  cross-validated RMSE = 8.796
maxDepth= 5  maxIter= 10  cross-validated RMSE = 8.802
maxDepth= 3  maxIter= 30  cross-validated RMSE = 8.884
maxDepth= 3  maxIter= 20  cross-validated RMSE = 8.945
maxDepth= 4  maxIter= 10  cross-validated RMSE = 9.109
maxDepth= 3  maxIter= 10  cross-validated RMSE = 9.233


### Step 3: reading the grid

Cross-validated RMSE ($, lower is better):

| maxDepth \ maxIter | 10 | 20 | 30 |
| --- | --- | --- | --- |
| 3 | 9.233 | 8.945 | 8.884 |
| 4 | 9.109 | 8.746 | 8.632 |
| 5 | 8.802 | 8.681 | **8.615** |
| 6 | 8.796 | 8.780 | 8.795 |

* **Best setting:** maxDepth = 5, maxIter = 30 (CV RMSE $8.615).
* **Shape:** the best depth is now inside the grid: at 30 rounds, depths 4 and 6 are both worse than 5 ($8.632 and $8.795). Shallow trees need more rounds: at depths 3, 4 and 5 every extra 10 rounds lowers the error, while at depth 6 the score does not change with more rounds ($8.796, $8.780, $8.795).
* **Edge:** maxIter = 30 is the top of its range, but going from 20 to 30 rounds at depth 5 gained only $0.066, so more rounds would pay little.
* **Spread:** best to worst is $0.618 (8.615 against 9.233 for depth 3 with 10 rounds), about 7%. The top five settings are within $0.165 of each other, so the choice among them is close; tuning mainly helped by avoiding shallow trees with few rounds.

### Step 4: the final test, once

`CrossValidator` refits the best setting on all of `fare_train` and keeps it as `bestModel`. Only now is `fare_test` used, and only once. The training score next to it shows how much the model overfits: a large gap between the two means it has memorised training trips instead of learning fares.

In [0]:
tuned_model = cv_model2.bestModel          # PipelineModel, refitted on the whole training set
best_gbt = tuned_model.stages[-1]
print(f"best setting: maxDepth={best_gbt.getMaxDepth()}, maxIter={best_gbt.getMaxIter()}")

pred_tuned_train = tuned_model.transform(fare_train)
pred_tuned_test = tuned_model.transform(fare_test)

print("train:", regression_report(pred_tuned_train))
print("test :", regression_report(pred_tuned_test))

best setting: maxDepth=5, maxIter=30
train: {'rmse': 7.329, 'mae': 3.872, 'r2': 0.837}
test : {'rmse': 7.862, 'mae': 4.112, 'r2': 0.816}


**Reading the final result.**

* **Against the hand-set models:** tuned GBT at pickup has RMSE $7.86 on the test set (R² 0.816), against $11.30 for the hand-set GBT with one-hot encoding and $10.70 for linear regression.
* **Against the zone-pair lookup:** the lookup has RMSE $8.49, MAE $3.98 and R² 0.785. The tuned model beats it on RMSE ($7.86) and R² (0.816), but is slightly worse on MAE ($4.11 against $3.98). The lookup is about as accurate on a typical trip, and the model's advantage is fewer large misses.
* **Overfitting:** training RMSE is $7.33 against $7.86 on the test set, a gap of $0.53 (about 7% of the test RMSE). The gap is small, so the model generalises rather than memorising training trips.
* **Validation versus test:** the cross-validated RMSE from Step 3 ($8.615) is higher than the test RMSE ($7.86). A plausible reason, not tested here: each cross-validation model trains on only two-thirds of `fare_train`, while the final model trains on all of it. The two numbers are of similar size, so the validation was a reasonable guide.

## Project A: saving models and tracking runs

Tuning is done and the results are stable, so this is the moment to freeze them. Any later change (for example a zone-pair feature) can then be compared with an exact baseline instead of numbers copied by hand.

- **Saving** stores the whole fitted `PipelineModel`: the `StringIndexer` and the GBT together, so a loaded model can predict from raw columns.
- **Tracking** with MLflow records every model as a *run*: its parameters (settings), metrics (scores on `fare_test`), tags (labels for filtering) and, for the tuned GBT, the saved model files. Runs of one notebook are grouped in an *experiment*.

### Step 1: save the tuned model, reload it, and check

A saved model is only proven once it has been reloaded and gives the **same predictions** as the one in memory. The checks below compare 500 test trips, then the whole test set. The folder size matters because serverless compute limits a single Spark ML model to 100 MB.

In [0]:
MODEL_DIR = "/Volumes/taxi/taxi_ml/ml_models/fare_a1_gbt_native"

tuned_model.write().overwrite().save(MODEL_DIR)
reloaded_model = PipelineModel.load(MODEL_DIR)

# The same 500 raw test trips go through both models
check_pdf = fare_test.limit(500).toPandas()
check_df = spark.createDataFrame(check_pdf)

pred_memory = tuned_model.transform(check_df).select("prediction").toPandas()["prediction"]
pred_reloaded = reloaded_model.transform(check_df).select("prediction").toPandas()["prediction"]

max_diff = float((pred_memory - pred_reloaded).abs().max())
size_mb = sum(os.path.getsize(os.path.join(root, f))
              for root, _, files in os.walk(MODEL_DIR) for f in files) / 1e6

print(f"rows compared: {len(check_pdf)} | largest prediction difference: {max_diff:.2e} | model folder: {size_mb:.2f} MB")

rows compared: 500 | largest prediction difference: 0.00e+00 | model folder: 0.16 MB


In [0]:
# Full-test-set check: the reloaded model must reproduce the tuned model's scores exactly
print("in memory:", regression_report(pred_tuned_test))
print("reloaded :", regression_report(reloaded_model.transform(fare_test)))

in memory: {'rmse': 7.862, 'mae': 4.112, 'r2': 0.816}
reloaded : {'rmse': 7.862, 'mae': 4.112, 'r2': 0.816}


In [0]:
manifest_a = write_manifest(
    MODEL_DIR, "fare_a1_gbt_native",
    tables=["taxi.taxi_ml.fare_train", "taxi.taxi_ml.fare_test"],
    feature_df=fare_train, categorical=fare_cat_cols, numeric=fare_num_pickup,
    split={"method": "random 80/20 of the 2% sample", "seed": 42,
           "train_table": "taxi.taxi_ml.fare_train", "test_table": "taxi.taxi_ml.fare_test"},
    hyperparameters={"model": "GBTRegressor", "maxDepth": best_gbt.getMaxDepth(), "maxIter": best_gbt.getMaxIter(),
                     "stepSize": best_gbt.getStepSize(), "maxBins": best_gbt.getMaxBins()},
    metrics=regression_report(pred_tuned_test),
)
print(json.dumps({k: manifest_a[k] for k in ["source_silver_table", "input_tables", "metrics", "software"]}, indent=2))

{
  "source_silver_table": {
    "table": "taxi.taxi_silver.yellow_taxi_trips_cleaned",
    "delta_version": 39,
    "last_write": "2026-11-23 08:02:16",
    "rows": 12245631
  },
  "input_tables": [
    {
      "table": "taxi.taxi_ml.fare_train",
      "delta_version": 34,
      "last_write": "2026-11-28 08:41:19",
      "rows": 195776
    },
    {
      "table": "taxi.taxi_ml.fare_test",
      "delta_version": 34,
      "last_write": "2026-11-28 08:41:23",
      "rows": 48730
    }
  ],
  "metrics": {
    "rmse": 7.862,
    "mae": 4.112,
    "r2": 0.816
  },
  "software": {
    "python": "3.12.3",
    "spark": "4.2.0",
    "mlflow": "3.12.0"
  }
}


### Step 2: track every model as an MLflow run

One run per model, all with the same tags (`task=fare`, split, seed, `features=A1/A2/none`) so later experiments can be filtered and compared in one table. Scores are recomputed from the prediction DataFrames, and hyperparameters are read from the model objects, so nothing is typed in by hand. Only the tuned GBT also gets its model files attached, because refitting the older models just to store them would waste compute.

In [0]:
COMMON_TAGS = {"task": "fare", "split": "fare_train / fare_test", "seed": "42", "sample": "2% of Silver"}

def delete_runs_named(name, task="fare"):
    """Remove earlier runs of this task with the same name, so re-running a logging cell does not create duplicates."""
    old = mlflow.search_runs(filter_string=f"tags.task = '{task}' and tags.mlflow.runName = '{name}'")
    for run_id in old["run_id"]:
        mlflow.delete_run(run_id)

# MLflow cannot compute a dataset profile on serverless (needs RDDs); the input is still recorded,
# so silence that warning instead of printing it for every run.
logging.getLogger("mlflow.data.spark_dataset").setLevel(logging.ERROR)

def log_lineage(tables):
    """Record the Delta tables a run read (name + version) as MLflow dataset inputs. Never raises."""
    with warnings.catch_warnings():
        # MLflow's "integer columns" schema hint: harmless here, we never serve these tables as a model input
        warnings.simplefilter("ignore", category=UserWarning)
        for t in tables:
            try:
                snap = table_snapshot(t)
                ds = mlflow.data.from_spark(spark.table(t), name=t.split(".")[-1],
                                            table_name=t, version=str(snap["delta_version"]))
                mlflow.log_input(ds, context="training")
            except Exception as e:
                print(f"  (lineage skipped for {t}: {type(e).__name__})")

def log_run(run_name, params, metrics, features, model_dir=None, extra_tags=None, tags=None,
            data_tables=None, figures=None):
    """Log one run: parameters, metrics on the test set, tags and (optionally) the saved model folder.
    Project A uses COMMON_TAGS; other projects pass their own tags (the task tag keeps them apart).
    Every run also logs the Delta version and row count of its data tables (DATA_TABLES by task)."""
    tags = tags or COMMON_TAGS
    tables = data_tables or DATA_TABLES[tags["task"]]
    delete_runs_named(run_name, tags["task"])
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_params(data_params(tables))
        log_lineage(tables)                                   # which tables, at which version
        mlflow.log_metrics(metrics)
        mlflow.set_tags({**tags, "features": features, **(extra_tags or {})})
        if model_dir:
            mlflow.log_artifacts(model_dir, artifact_path="model")
        for file_name, figure in (figures or {}).items():
            mlflow.log_figure(figure, f"figures/{file_name}")

# (run name, predictions on fare_test, feature set, parameters)
runs_to_log = [
    ("baseline_training_average", baseline_pred, "none",
     {"model": "constant (training mean)"}),
    ("baseline_zone_pair_average", pair_baseline_pred, "A1",
     {"model": "lookup: mean fare per pickup/drop-off zone pair"}),
    ("linear_regression_pickup", pred_lr_pickup, "A1",
     {"model": "LinearRegression", "encoding": "one-hot", "regParam": fare_lr.getRegParam(),
      "elasticNetParam": fare_lr.getElasticNetParam(), "maxIter": fare_lr.getMaxIter()}),
    ("random_forest_pickup", pred_rf_pickup, "A1",
     {"model": "RandomForestRegressor", "encoding": "one-hot", "numTrees": fare_rf.getNumTrees(),
      "maxDepth": fare_rf.getMaxDepth()}),
    ("gbt_pickup", pred_gbt_pickup, "A1",
     {"model": "GBTRegressor", "encoding": "one-hot", "maxIter": fare_gbt.getMaxIter(),
      "maxDepth": fare_gbt.getMaxDepth(), "stepSize": fare_gbt.getStepSize()}),
    ("linear_regression_dropoff", pred_lr_drop, "A2",
     {"model": "LinearRegression", "encoding": "one-hot", "regParam": fare_lr.getRegParam(),
      "elasticNetParam": fare_lr.getElasticNetParam(), "maxIter": fare_lr.getMaxIter()}),
    ("random_forest_dropoff", pred_rf_drop, "A2",
     {"model": "RandomForestRegressor", "encoding": "one-hot", "numTrees": fare_rf.getNumTrees(),
      "maxDepth": fare_rf.getMaxDepth()}),
    ("gbt_dropoff", pred_gbt_drop, "A2",
     {"model": "GBTRegressor", "encoding": "one-hot", "maxIter": fare_gbt.getMaxIter(),
      "maxDepth": fare_gbt.getMaxDepth(), "stepSize": fare_gbt.getStepSize()}),
]

for run_name, pred_df, features, params in runs_to_log:
    log_run(run_name, params, regression_report(pred_df), features)

# The tuned model: parameters, how it was chosen, and the saved model files
log_run(
    "gbt_tuned_native_categories",
    params={
        "model": "GBTRegressor",
        "encoding": "category indices (no one-hot)",
        "maxDepth": best_gbt.getMaxDepth(),
        "maxIter": best_gbt.getMaxIter(),
        "stepSize": best_gbt.getStepSize(),
        "maxBins": best_gbt.getMaxBins(),
        "cv_folds": 3,
        "cv_rmse": round(cv_results2[0][2], 3),
        "tuning_grids": ("grid 1: maxDepth [6,8,10] x maxIter [20,40] -> best (6, 20), CV RMSE 8.780; "
                         "grid 2: maxDepth [3,4,5,6] x maxIter [10,20,30] -> best (5, 30), CV RMSE 8.615"),
    },
    metrics=regression_report(pred_tuned_test),
    features="A1",
    model_dir=MODEL_DIR,
    extra_tags={"stage": "tuned"},
)

### Step 3: compare all runs in one table

Every later experiment, such as a zone-pair feature, is logged the same way and appears in this table, sorted by test RMSE.

In [0]:
runs = mlflow.search_runs(filter_string="tags.task = 'fare'", order_by=["start_time DESC"])

duplicates = runs[runs.duplicated(subset="tags.mlflow.runName", keep="first")]   # keeps the newest of each name
for run_id in duplicates["run_id"]:
    mlflow.delete_run(run_id)
    
runs = mlflow.search_runs(filter_string="tags.task = 'fare'", order_by=["metrics.rmse ASC"])
print(runs[["tags.mlflow.runName", "tags.features", "metrics.rmse", "metrics.mae", "metrics.r2"]]
      .to_string(index=False))

                     tags.mlflow.runName tags.features  metrics.rmse  metrics.mae  metrics.r2
                             gbt_dropoff            A2         5.372        1.841       0.914
                   random_forest_dropoff            A2         5.559        1.934       0.908
               linear_regression_dropoff            A2         5.737        2.594       0.902
             gbt_tuned_native_categories            A1         7.862        4.112       0.816
gbt_tuned_native_categories_pair_feature       A1+pair         7.891        3.757       0.814
              baseline_zone_pair_average            A1         8.493        3.981       0.785
                linear_regression_pickup            A1        10.697        6.692       0.659
                              gbt_pickup            A1        11.301        7.331       0.619
                    random_forest_pickup            A1        11.872        7.810       0.580
               baseline_training_average          none      

### Step 4: log the tuning searches as nested runs

So far every *finished model* has a run. The *search* that chose its settings does not: the CV scores of the
other settings only live in Python lists. Here each search becomes one **parent run** with one **child run
per setting tried**, built from the CV results we already computed (nothing is retrained).

Search runs carry the tag `search_task` instead of `task`, so the model comparison tables above and
below never include them.

In [0]:
def delete_search_runs(search_name):
    """Delete a search's parent run AND all its child runs (they share the search_name tag)."""
    found = mlflow.search_runs(
        filter_string=f"tags.search_name = '{search_name}'",
        output_format="list",
    )
    for r in found:
        mlflow.delete_run(r.info.run_id)
    return len(found)


def log_search(search_name, task, rows, metric_name, higher_is_better, fixed_params=None):
    """
    One parent run for the whole search, one nested child run per tried setting.
    rows: list of {"params": {...}, "metric": float, "std": float or None}
    """
    delete_search_runs(search_name)                      # re-running the cell never duplicates
    pick = max if higher_is_better else min
    best = pick(rows, key=lambda r: r["metric"])
    tags = {"search_name": search_name, "search_task": task}

    with mlflow.start_run(run_name=search_name):
        mlflow.set_tags({**tags, "kind": "search"})
        mlflow.log_params({
            **(fixed_params or {}),
            **data_params(DATA_TABLES[task]),            # same data version as the model runs
            "n_settings": len(rows),
        })
        for row in rows:
            label = ", ".join(f"{k}={v}" for k, v in row["params"].items())
            with mlflow.start_run(run_name=label, nested=True):
                mlflow.set_tags({**tags, "kind": "trial"})
                mlflow.log_params(row["params"])
                mlflow.log_metric(metric_name, row["metric"])
                if row.get("std") is not None:
                    mlflow.log_metric(metric_name + "_std", row["std"])
        mlflow.log_metric("best_" + metric_name, best["metric"])
        mlflow.set_tag("best_setting", ", ".join(f"{k}={v}" for k, v in best["params"].items()))
    return len(rows)

In [0]:
# cv_results / cv_results2 hold tuples (maxDepth, maxIter, cv_rmse) from the two search rounds
def rows_from_tuples(results):
    return [{"params": {"maxDepth": d, "maxIter": i}, "metric": float(r)} for d, i, r in results]

n1 = log_search("fare_gbt_search_round1", "fare", rows_from_tuples(cv_results),
                metric_name="cv_rmse", higher_is_better=False,
                fixed_params={"model": "GBTRegressor"})
n2 = log_search("fare_gbt_search_round2", "fare", rows_from_tuples(cv_results2),
                metric_name="cv_rmse", higher_is_better=False,
                fixed_params={"model": "GBTRegressor"})
print(f"logged {n1} + {n2} child runs")

logged 6 + 12 child runs


### What was saved

- **Model:** `/Volumes/taxi/taxi_ml/ml_models/fare_a1_gbt_native` holds the tuned GBT with its preprocessing (`PipelineModel`). Reload it with `PipelineModel.load(path)`; the reload was checked against the in-memory model.
- **Runs:** nine MLflow runs (two baselines, six untuned models, one tuned model) with identical tags, so any future model can be compared with them. The two tuning searches are stored separately as nested runs (Step 4: 6 + 12 settings), under the `search_task` tag.
- **Baseline for the next step:** tuned GBT at pickup, test RMSE $7.862, MAE $4.112, R² 0.816.
- **Manifest:** `manifest.json` in the model folder records the Delta versions and row counts of the data used, the features with their types, the split, the settings, the test metrics and the software versions. Every MLflow run also logs the data versions as parameters and lists its input tables as dataset inputs.

## Project A: a zone-pair feature

The zone-pair lookup (RMSE $8.49) was strong because fares depend on the *route*, which separate zone columns cannot express. This section hands the model that information as a feature: `pair_mean_fare`, the average training fare of the trip's pickup → drop-off pair.

### The leakage problem

If a trip's own fare is part of its pair's average, the feature contains the answer. Training scores then look excellent and the test score disappoints. Two precautions:

- **Out-of-fold averages for training trips.** The training set is split into 5 folds; a trip's feature is the pair average computed from the *other* 4 folds, so it never contains its own fare. (Leave-one-out averages are not safe: within a pair, the feature moves in exact opposition to the fare, and a tree can decode it.)
- **Smoothing.** `(sum + 5 × overall mean) / (count + 5)` pulls pairs with few trips towards the overall mean. A pair never seen in training simply gets the overall mean.

Test trips use the average over the whole training set. The test set's own fares are never used in any average.

### Step 1: build the feature and freeze it

In [0]:
N_FOLDS = 5         # each training trip gets the pair average computed from the other folds
SMOOTHING = 5       # a pair with few trips is pulled towards the overall average
PAIR = ["PULocationID", "DOLocationID"]

# Overall average fare of the training set: the fallback for pairs with no training trips
prior = fare_train.agg(F.avg("fare_amount")).first()[0]

# Random fold number for every training trip; saved so the folds cannot change between uses
(fare_train.withColumn("te_fold", (F.rand(seed=42) * N_FOLDS).cast("int"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("taxi.taxi_ml.fare_train_folds"))
train_folds = spark.table("taxi.taxi_ml.fare_train_folds")

# Per pair: total fare and trip count over the whole training set, and within each fold
pair_stats = (train_folds.groupBy(PAIR)
              .agg(F.sum("fare_amount").alias("pair_sum"), F.count("*").alias("pair_n")))
pair_fold_stats = (train_folds.groupBy(PAIR + ["te_fold"])
                   .agg(F.sum("fare_amount").alias("fold_sum"), F.count("*").alias("fold_n")))

# Training trips: pair average from the OTHER folds (whole-pair totals minus this trip's fold), smoothed
train_pair_df = (
    train_folds
    .join(pair_stats, on=PAIR, how="left")
    .join(pair_fold_stats, on=PAIR + ["te_fold"], how="left")
    .withColumn("pair_mean_fare",
                (F.col("pair_sum") - F.col("fold_sum") + SMOOTHING * F.lit(float(prior)))
                / (F.col("pair_n") - F.col("fold_n") + SMOOTHING))
    .drop("pair_sum", "pair_n", "fold_sum", "fold_n", "te_fold")
)

# Test trips: pair average over the whole training set, smoothed; unseen pairs get the overall mean
test_pair_df = (
    fare_test
    .join(pair_stats, on=PAIR, how="left")
    .withColumn("pair_mean_fare",
                (F.coalesce(F.col("pair_sum"), F.lit(0.0)) + SMOOTHING * F.lit(float(prior)))
                / (F.coalesce(F.col("pair_n"), F.lit(0)) + SMOOTHING))
    .drop("pair_sum", "pair_n")
)

# Freeze everything: the two datasets, and the lookup needed to score new trips later
train_pair_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.fare_train_pair")
test_pair_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.fare_test_pair")
pair_stats.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.pair_stats")

fare_train_pair = spark.table("taxi.taxi_ml.fare_train_pair")
fare_test_pair = spark.table("taxi.taxi_ml.fare_test_pair")

# Checks: same row counts as before, no missing values, and the leakage check
print("rows (train, test):", fare_train_pair.count(), fare_test_pair.count())
print("missing feature values (train, test):",
      fare_train_pair.filter(F.col("pair_mean_fare").isNull()).count(),
      fare_test_pair.filter(F.col("pair_mean_fare").isNull()).count())

corr_train = fare_train_pair.stat.corr("pair_mean_fare", "fare_amount")
corr_test = fare_test_pair.stat.corr("pair_mean_fare", "fare_amount")
print(f"correlation of pair_mean_fare with the fare: train {corr_train:.3f} | test {corr_test:.3f}")

rows (train, test): 195776 48730
missing feature values (train, test): 0 0
correlation of pair_mean_fare with the fare: train 0.862 | test 0.869


**Reading the leakage check.** The correlation of `pair_mean_fare` with the fare is 0.862 on the training set and 0.869 on the test set. They are close, so the out-of-fold averages do not leak the training fares. A feature that contained its own row's fare would score clearly higher on the training set than on the test set.

### Step 2: does the feature help? (validation, test set untouched)

The tuned setting (maxDepth 5, maxIter 30) is cross-validated on `fare_train_pair` twice: once with the original pickup features and once with `pair_mean_fare` added. Same folds, same model, only the feature differs, so any difference in RMSE is due to the feature. A difference smaller than about $0.17, the spread between the best tuning settings in the grid search, would not justify the extra complexity.

In [0]:
fare_num_pair = fare_num_pickup + ["pair_mean_fare"]


def cross_validated_rmse(numeric_cols, data):
    """3-fold cross-validated RMSE of the tuned GBT setting, using the given numeric features."""
    gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount", maxBins=300,
                       maxDepth=best_gbt.getMaxDepth(), maxIter=best_gbt.getMaxIter(),
                       stepSize=0.1, seed=42)
    pipeline = Pipeline(stages=build_tree_stages(fare_cat_cols, numeric_cols) + [gbt])
    grid = ParamGridBuilder().addGrid(gbt.maxDepth, [best_gbt.getMaxDepth()]).build()     # one setting
    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=grid,
        evaluator=RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse"),
        numFolds=3, parallelism=2, seed=42,
    )
    return cv.fit(data).avgMetrics[0]


cv_without_pair = cross_validated_rmse(fare_num_pickup, fare_train_pair)
cv_with_pair = cross_validated_rmse(fare_num_pair, fare_train_pair)

print(f"cross-validated RMSE without the pair feature: {cv_without_pair:.3f}")
print(f"cross-validated RMSE with the pair feature   : {cv_with_pair:.3f}")

cross-validated RMSE without the pair feature: 8.615
cross-validated RMSE with the pair feature   : 8.399


**Reading the comparison.** Cross-validated RMSE is $8.615 without the pair feature and $8.399 with it, a difference of $0.216. This is slightly above the $0.17 tuning spread, so on validation the feature looks marginally useful. This number is slightly optimistic, because the folds used to build the feature differ from the cross-validation folds, so the test set in Step 3 is the deciding evidence.

### Step 3: the final test, once

The model with `pair_mean_fare` is fitted on all of `fare_train_pair` and scored on `fare_test_pair`, once. It receives the lookup as an input, so it should beat the lookup itself (RMSE $8.49, MAE $3.98) on **both** RMSE and MAE. If it does not, the feature is not being used well and the numbers need a closer look.

In [0]:
gbt_pair = GBTRegressor(featuresCol="features", labelCol="fare_amount", maxBins=300,
                        maxDepth=best_gbt.getMaxDepth(), maxIter=best_gbt.getMaxIter(),
                        stepSize=0.1, seed=42)

model_pair = Pipeline(stages=build_tree_stages(fare_cat_cols, fare_num_pair) + [gbt_pair]).fit(fare_train_pair)

pred_pair_train = model_pair.transform(fare_train_pair)
pred_pair_test = model_pair.transform(fare_test_pair)

print("train:", regression_report(pred_pair_train))
print("test :", regression_report(pred_pair_test))

train: {'rmse': 7.055, 'mae': 3.505, 'r2': 0.849}
test : {'rmse': 7.891, 'mae': 3.757, 'r2': 0.814}


**Reading the final result.**

- **Test performance:** RMSE $7.891, MAE $3.757, R² 0.814, against $7.862 / $4.112 / 0.816 for the tuned GBT without the feature and $8.493 / $3.981 / 0.785 for the zone-pair lookup. The model with the feature beats the lookup on both RMSE and MAE, as it should.
- **What the feature changed:** MAE improves by $0.35 (about 9%), so the typical fare error is smaller. RMSE is unchanged ($0.03 worse, well within the tuning spread), so the large errors are not reduced. A route average tells the model what a pair usually costs, but the big misses come from trips whose fare depends on things unknown at pickup, such as the final distance and duration.
- **Overfitting:** training RMSE $7.055 against $7.891 on the test set (gap $0.84). The gap is larger than without the feature (0.53), so the model partly relies on the pair average.
- **Validation versus test:** cross-validation suggested a $0.22 RMSE gain, which the test did not confirm. The test set is the reference.
- **Decision:** the pair feature is worth keeping if the typical error (MAE) matters, and not worth the extra complexity if RMSE is the target. It is not a large improvement, which fits the earlier observation that further tuning would not help much without better pickup information.

### Step 4: save, log, and score new trips

The saved model expects `pair_mean_fare` as an input, so scoring a **new** trip needs two steps: look up its pair in the saved `pair_stats` table, then apply the model. `add_pair_feature` does the first step with the same smoothing and fallback as in training, and the check below proves the whole route from raw trips to predictions on the saved model.

In [0]:
MODEL_DIR_PAIR = "/Volumes/taxi/taxi_ml/ml_models/fare_a1_gbt_pair"
model_pair.write().overwrite().save(MODEL_DIR_PAIR)
reloaded_pair = PipelineModel.load(MODEL_DIR_PAIR)


def add_pair_feature(df):
    """Add pair_mean_fare to RAW trips, using the saved training-set pair statistics."""
    return (
        df.join(spark.table("taxi.taxi_ml.pair_stats"), on=PAIR, how="left")
        .withColumn("pair_mean_fare",
                    (F.coalesce(F.col("pair_sum"), F.lit(0.0)) + SMOOTHING * F.lit(float(prior)))
                    / (F.coalesce(F.col("pair_n"), F.lit(0)) + SMOOTHING))
        .drop("pair_sum", "pair_n")
    )


# Raw test trips -> lookup -> reloaded model must reproduce the in-memory scores exactly
print("in memory:", regression_report(pred_pair_test))
print("reloaded :", regression_report(reloaded_pair.transform(add_pair_feature(fare_test))))

in memory: {'rmse': 7.891, 'mae': 3.757, 'r2': 0.814}
reloaded : {'rmse': 7.891, 'mae': 3.757, 'r2': 0.814}


In [0]:
manifest_pair = write_manifest(
    MODEL_DIR_PAIR, "fare_a1_gbt_pair",
    tables=["taxi.taxi_ml.fare_train_pair", "taxi.taxi_ml.fare_test_pair", "taxi.taxi_ml.pair_stats"],
    feature_df=fare_train_pair, categorical=fare_cat_cols, numeric=fare_num_pair,
    split={"method": "same random 80/20 split as fare_a1_gbt_native", "seed": 42,
           "note": "pair_mean_fare is built out-of-fold on the training rows only"},
    hyperparameters={"model": "GBTRegressor", "maxDepth": gbt_pair.getMaxDepth(), "maxIter": gbt_pair.getMaxIter(),
                     "stepSize": gbt_pair.getStepSize(), "maxBins": gbt_pair.getMaxBins(),
                     "te_folds": N_FOLDS, "smoothing": SMOOTHING, "pair_prior": round(prior, 4)},
    metrics=regression_report(pred_pair_test),
    extra={"scoring_note": "new trips need pair_mean_fare from taxi.taxi_ml.pair_stats before scoring"},
)
print(json.dumps({k: manifest_pair[k] for k in ["input_tables", "metrics"]}, indent=2))

{
  "input_tables": [
    {
      "table": "taxi.taxi_ml.fare_train_pair",
      "delta_version": 8,
      "last_write": "2026-11-28 09:07:07",
      "rows": 195776
    },
    {
      "table": "taxi.taxi_ml.fare_test_pair",
      "delta_version": 8,
      "last_write": "2026-11-28 09:07:10",
      "rows": 48730
    },
    {
      "table": "taxi.taxi_ml.pair_stats",
      "delta_version": 8,
      "last_write": "2026-11-28 09:07:12",
      "rows": 13263
    }
  ],
  "metrics": {
    "rmse": 7.891,
    "mae": 3.757,
    "r2": 0.814
  }
}


In [0]:
log_run(
    "gbt_tuned_native_categories_pair_feature",
    params={
        "model": "GBTRegressor",
        "encoding": "category indices (no one-hot)",
        "maxDepth": gbt_pair.getMaxDepth(),
        "maxIter": gbt_pair.getMaxIter(),
        "stepSize": gbt_pair.getStepSize(),
        "maxBins": gbt_pair.getMaxBins(),
        "pair_feature": "out-of-fold smoothed pair mean fare",
        "te_folds": N_FOLDS,
        "smoothing": SMOOTHING,
        "pair_prior": round(prior, 4),
        "cv_rmse_without_feature": round(cv_without_pair, 3),
        "cv_rmse_with_feature": round(cv_with_pair, 3),
    },
    metrics=regression_report(pred_pair_test),
    features="A1+pair",
    model_dir=MODEL_DIR_PAIR,
    data_tables=["taxi.taxi_ml.fare_train_pair", "taxi.taxi_ml.fare_test_pair", "taxi.taxi_ml.pair_stats"],
    extra_tags={"stage": "feature experiment"},
)

runs = mlflow.search_runs(filter_string="tags.task = 'fare'", order_by=["metrics.rmse ASC"])
print(runs[["tags.mlflow.runName", "tags.features", "metrics.rmse", "metrics.mae", "metrics.r2"]]
      .to_string(index=False))

                     tags.mlflow.runName tags.features  metrics.rmse  metrics.mae  metrics.r2
                             gbt_dropoff            A2         5.372        1.841       0.914
                   random_forest_dropoff            A2         5.559        1.934       0.908
               linear_regression_dropoff            A2         5.737        2.594       0.902
             gbt_tuned_native_categories            A1         7.862        4.112       0.816
gbt_tuned_native_categories_pair_feature       A1+pair         7.891        3.757       0.814
              baseline_zone_pair_average            A1         8.493        3.981       0.785
                linear_regression_pickup            A1        10.697        6.692       0.659
                              gbt_pickup            A1        11.301        7.331       0.619
                    random_forest_pickup            A1        11.872        7.810       0.580
               baseline_training_average          none      

### What was added

- **Model:** `/Volumes/taxi/taxi_ml/ml_models/fare_a1_gbt_pair` holds the GBT with the pair feature. It needs `pair_mean_fare` as input, so new trips go through `add_pair_feature` (which reads the saved `pair_stats` table) first.
- **Run:** a tenth MLflow run, `gbt_tuned_native_categories_pair_feature`, logged with the same tags.
- **Tables:** `fare_train_folds`, `fare_train_pair`, `fare_test_pair` and `pair_stats` in `taxi.taxi_ml`.
- **Manifest:** `manifest.json` in the model folder, with the data versions of `fare_train_pair`, `fare_test_pair` and `pair_stats`, the features, settings, metrics and software versions.

## Project A: results and conclusions

### Results: all models on the test set

| Model | Features | RMSE ($) | MAE ($) | R² |
|---|---|---:|---:|---:|
| Baseline: training average | none | 18.316 | 12.210 | 0.000 |
| Baseline: zone-pair average | pickup and drop-off zone | 8.493 | 3.981 | 0.785 |
| Linear regression | A1: pickup | 10.697 | 6.692 | 0.659 |
| Random forest | A1: pickup | 11.872 | 7.810 | 0.580 |
| Gradient-boosted trees | A1: pickup | 11.301 | 7.331 | 0.619 |
| Gradient-boosted trees, tuned, native categories | A1: pickup | 7.862 | 4.112 | 0.816 |
| Gradient-boosted trees, tuned, native categories, with pair feature | A1+pair: pickup | 7.891 | 3.757 | 0.814 |
| Linear regression | A2: drop-off | 5.737 | 2.594 | 0.902 |
| Random forest | A2: drop-off | 5.559 | 1.934 | 0.908 |
| Gradient-boosted trees | A2: drop-off | **5.372** | **1.841** | **0.914** |

- **RMSE:** root of the mean squared error; penalises large errors most.
- **MAE:** average absolute error, in dollars.
- **R²:** share of the variance in fares explained (0 = no better than the average).

### Look inside the model: feature importance and errors

In [0]:
# Most important features of the drop-off random forest (names come from the feature_names helper)
rf_names = feature_names(pred_rf_drop)
rf_importance = model_rf_drop.stages[-1].featureImportances.toArray()

for name, value in sorted(zip(rf_names, rf_importance), key=lambda pair: -pair[1])[:10]:
    print(f"{name:35s}{value:.4f}")

# Where is the model wrong? Mean absolute error by trip type
(pred_rf_drop
    .withColumn("abs_err", F.abs(F.col("fare_amount") - F.col("prediction")))
    .groupBy("trip_type")
    .agg(F.count("*").alias("trips"), F.round(F.avg("abs_err"), 2).alias("mae"))
    .show())

trip_duration_mins                 0.3479
trip_distance                      0.3354
trip_type_vec_airport              0.0868
trip_type_vec_standard             0.0834
PULocationID_vec_132               0.0601
DOLocationID_vec_265               0.0442
PULocationID_vec_138               0.0135
DOLocationID_vec_1                 0.0107
DOLocationID_vec_132               0.0097
DOLocationID_vec_138               0.0017
+---------+-----+----+
|trip_type|trips| mae|
+---------+-----+----+
| standard|43623|1.53|
|  airport| 5107|5.39|
+---------+-----+----+



In [0]:
# The ten largest errors of the best model: outliers (e.g. very long trips) or a pattern?
(pred_gbt_drop
    .withColumn("abs_err", F.abs(F.col("fare_amount") - F.col("prediction")))
    .select("trip_type", "PULocationID", "DOLocationID", "trip_distance", "trip_duration_mins",
            "fare_amount", F.round("prediction", 2).alias("prediction"),
            F.round("abs_err", 2).alias("abs_err"))
    .orderBy(F.col("abs_err").desc())
    .show(10))

+---------+------------+------------+-------------+------------------+-----------+----------+-------+
|trip_type|PULocationID|DOLocationID|trip_distance|trip_duration_mins|fare_amount|prediction|abs_err|
+---------+------------+------------+-------------+------------------+-----------+----------+-------+
|  airport|         132|         265|        78.87|            112.25|      540.6|    202.82| 337.78|
| standard|         116|         265|        16.35|             36.75|      400.0|     77.77| 322.23|
|  airport|         138|         265|        58.04|             88.63|      382.4|    172.57| 209.83|
|  airport|         132|         265|         21.4|              74.9|       91.9|    269.66| 177.76|
|  airport|         132|         265|         28.4|             72.43|      110.8|    269.59| 158.79|
| standard|         164|         265|        22.22|             43.88|       2.25|    138.09| 135.84|
|  airport|         132|         265|         9.79|            107.52|      179.4|

### Conclusions for Project A

- **Pickup information alone explains a good share of the fare.** The linear model cuts RMSE from $18.32 (average baseline) to $10.70 (R² 0.659).
- **A plain zone-pair lookup beats every untuned pickup model.** Predicting the average training fare of each pickup → drop-off pair gives RMSE $8.49 (R² 0.785), better than all three A1 models with one-hot zones. A plausible cause, not tested here: with separate one-hot columns for the two zones, a model adds a pickup effect and a drop-off effect but cannot easily represent the specific route, and fares depend on the route. The tuned GBT with native categories does overtake the lookup on RMSE and R², but not on MAE.
- **With one-hot encoding, both tree models are worse than linear regression at pickup** (R² 0.580 for the forest and 0.619 for GBT, against 0.659). A likely cause is the ~250 one-hot columns per zone type: shallow trees can isolate only a few zones, while the linear model gives every zone its own coefficient. Step 1 of the tuning section supports this: on a validation split, native categories gave RMSE $9.114 against $11.660 for one-hot encoding.
- **Native categories plus tuning make GBT the best pickup model by RMSE and R².** Cross-validation chose maxDepth 5 and maxIter 30 (CV RMSE $8.615). On the test set it scores RMSE $7.86, MAE $4.11 and R² 0.816, ahead of linear regression ($10.70) and the zone-pair lookup ($8.49) on RMSE and R². Its MAE is slightly worse than the lookup's ($4.11 against $3.98), so on a typical trip the two are about equally accurate and the model's advantage is fewer large misses. Training RMSE is $7.33, a gap of about 7%, so it is not badly overfitted. Most of the gain came from the encoding, not the tuning: switching encodings moved the validation RMSE by about $2.5, while the best setting from the first, smaller grid (6/20) already scored the same on the test set ($7.86) and tuning moved the cross-validated score by $1.2 at most.
- **A zone-pair feature improves the typical error, not the large errors.** Adding the out-of-fold average fare of each pickup → drop-off pair (smoothed, built from training fares only) to the tuned GBT lowers MAE from $4.11 to $3.76 (about 9%), now below the lookup's $3.98, while RMSE stays at about $7.9 ($7.89 against $7.86, within the tuning spread). Cross-validation suggested a $0.22 RMSE gain, which the test set did not confirm, and the gap between training and test RMSE grew from $0.53 to $0.84. A plausible cause, not tested here: the remaining large errors come from information unavailable at pickup (final distance and duration), which a route average cannot supply, so the pickup model is probably close to its limit.
- **Adding distance and duration helps a lot.** RMSE falls to $5.37–5.74 and R² reaches 0.90–0.91, because the taximeter charges by distance and time. A2 is a ceiling, not a realistic forecast.
- **Gradient-boosted trees are the best model at drop-off on every metric** (RMSE $5.37, MAE $1.84, R² 0.914).
- **At drop-off, MAE is far below RMSE** (GBT: $1.84 vs $5.37). Most predictions are close, and a minority are far off. All ten of the largest errors of this model are trips ending in zone 265 ("Outside of NYC"), with fares such as $540.60 and $400.00 that the model cannot reproduce. One trip has a $2.25 fare for 22 miles, which looks like a recording error.
- **Errors concentrate in airport trips.** For the random forest at drop-off, MAE is $5.39 for airport trips against $1.53 for standard trips, so the long, expensive trips are where the models struggle most.

## Project B: will the rider tip 20% or more? (classification)

**Goal:** predict whether a credit-card trip ends with a tip of at least 20% of the fare (`high_tip = 1`).

- **Rows:** card payments only, because cash tips are not recorded
- **Split:** time-based, training on January–April and testing on May; Step 8 compares it with a random split
- **Metrics:** AUC and AUC-PR (threshold-free), accuracy and F1, always compared with the majority-class baseline

### Step 1: build the labelled dataset

Keep only credit-card trips (`payment_type = 1`) with a positive fare and a recorded `tip_pct`, then define the label `high_tip = 1` when `tip_pct >= 20`. The result is saved as `taxi.taxi_ml.tip_dataset`.

In [0]:
# Credit-card trips only (cash tips are not recorded), positive fare, tip_pct present.
# Label: high_tip = 1.0 when the tip is at least 20% of the fare.
(
    trips
    .filter((F.col("payment_type") == 1) & (F.col("fare_amount") > 0) & F.col("tip_pct").isNotNull())
    .withColumn("high_tip", (F.col("tip_pct") >= 20).cast("double"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("taxi.taxi_ml.tip_dataset")
)

tips = spark.table("taxi.taxi_ml.tip_dataset")
total = tips.count()

# Class balance. The larger share is the majority-class baseline: a model that always predicts
# the majority class already reaches that accuracy, so a real model has to beat it.
tips.groupBy("high_tip").count().withColumn("share", F.round(F.col("count") / total, 3)).show()

+--------+------+-----+
|high_tip| count|share|
+--------+------+-----+
|     0.0| 60892|0.283|
|     1.0|154149|0.717|
+--------+------+-----+



In [0]:
# Is the 20% cutoff cleanly placed? Count trips in 0.5-point bins around it.
(tips.filter(F.col("tip_pct").between(18.0, 22.0))
     .withColumn("bin", F.floor(F.col("tip_pct") * 2) / 2)
     .groupBy("bin").count().orderBy("bin").show(10))

+----+-----+
| bin|count|
+----+-----+
|18.0| 1018|
|18.5| 1807|
|19.0| 1098|
|19.5|  846|
|20.0| 2420|
|20.5| 1852|
|21.0| 2163|
|21.5| 2382|
|22.0|   17|
+----+-----+



**Checking the cutoff.** There is no pile-up just below 20%: the 19.5–20.0 bin holds 846 trips against 2,420 in the 20.0–20.5 bin, so rounding to cents is not pushing 20% tips into the low class, and the label `tip_pct >= 20` is kept. (The bin labelled 22.0 holds only trips at exactly 22.0%.)

### Step 2: features and a time-based split

The model is trained on **January–April** and tested on **May**. This mimics real use (predict the future from the past) and avoids a random split mixing neighbouring trips into both sets. Step 8 repeats the experiment with a random split to see whether it changes the picture.

Features are those listed in the task table of the setup (Step 4): fare, distance, duration, tolls, passengers, zones, trip type, congestion flag and the time features.

In [0]:
# Categorical (one-hot encoded) and numeric features
tip_cat_cols = ["PULocationID", "DOLocationID", "trip_type", "congestion_zone_flag"]
tip_num_cols = ["fare_amount", "trip_distance", "trip_duration_mins", "passenger_count", "tolls_amount",
                "is_weekend", "is_rush_hour", "hour_sin", "hour_cos"]

# Time-based split. pickup_month is a 'YYYY-MM' string, so comparing strings is the same as comparing months.
tip_train = tips.filter(F.col("pickup_month") < "2026-05")      # January-April
tip_test = tips.filter(F.col("pickup_month") == "2026-05")      # May

print(f"train {tip_train.count():,} | test {tip_test.count():,}")

train 166,653 | test 48,388


In [0]:
# Majority-class baseline on the test month: accuracy of always predicting "high tip".
# (AUC of any constant prediction is 0.5 by definition.)
may_high_share = tip_test.agg(F.avg("high_tip")).first()[0]
print(f"May baseline accuracy (always 'high tip'): {may_high_share:.3f}")

May baseline accuracy (always 'high tip'): 0.716


### Step 3: evaluation helper

| Metric | Meaning | Chance level |
|---|---|---|
| **AUC** (ROC) | Probability that a random high-tip trip is ranked above a random low-tip trip | 0.5 |
| **AUC-PR** | Area under the precision–recall curve | the share of positives (about 0.72 here), **not** 0.5 |
| **Accuracy** | Share of correct predictions | the majority-class share (about 0.72) |
| **F1** | Balance of precision and recall (weighted over both classes) | depends on the classes |

Because 72% of the trips are high-tip, accuracy and AUC-PR look good even for a useless model. AUC is the fairest headline number.

In [0]:
def classification_report(pred_df, label="high_tip"):
    """AUC, AUC-PR, accuracy and weighted F1 of a predictions DataFrame."""
    def binary(metric):
        return BinaryClassificationEvaluator(
            labelCol=label, rawPredictionCol="rawPrediction", metricName=metric
        ).evaluate(pred_df)

    def multiclass(metric):
        return MulticlassClassificationEvaluator(
            labelCol=label, predictionCol="prediction", metricName=metric
        ).evaluate(pred_df)

    return {
        "auc": round(binary("areaUnderROC"), 3),         # 0.5 = chance
        "auc_pr": round(binary("areaUnderPR"), 3),       # chance = share of positives
        "accuracy": round(multiclass("accuracy"), 3),
        "f1": round(multiclass("f1"), 3),                # weighted over both classes
    }

In [0]:
def threshold_report(pred_df, thresholds=(0.3, 0.4, 0.5, 0.6, 0.7), label="high_tip"):
    """Precision and recall for the 'high tip' class at several probability thresholds."""
    scored = pred_df.withColumn("p_high", vector_to_array("probability")[1])

    for t in thresholds:
        counts = scored.agg(
            F.sum(F.when((F.col("p_high") >= t) & (F.col(label) == 1), 1).otherwise(0)).alias("tp"),
            F.sum(F.when((F.col("p_high") >= t) & (F.col(label) == 0), 1).otherwise(0)).alias("fp"),
            F.sum(F.when((F.col("p_high") < t) & (F.col(label) == 1), 1).otherwise(0)).alias("fn"),
        ).first()

        precision = counts.tp / (counts.tp + counts.fp) if (counts.tp + counts.fp) else float("nan")
        recall = counts.tp / (counts.tp + counts.fn) if (counts.tp + counts.fn) else float("nan")
        print(f"threshold {t}: precision {precision:.3f} | recall {recall:.3f}")

### Step 4: logistic regression and a random forest

Both models use the same features. Logistic regression gets standardised numeric inputs (`scale=True`); the forest does not need scaling. Each is compared with the majority-class baseline from Step 2.

In [0]:
logreg = LogisticRegression(featuresCol="features", labelCol="high_tip", regParam=0.01, maxIter=100)
rf_clf = RandomForestClassifier(featuresCol="features", labelCol="high_tip", numTrees=30, maxDepth=8, seed=42)

# Logistic regression needs scaled numeric features (scale=True); the forest does not
model_logreg_time = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols, scale=True) + [logreg]
).fit(tip_train)
model_rf_time = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols) + [rf_clf]
).fit(tip_train)

pred_logreg_time = model_logreg_time.transform(tip_test)
pred_rf_time = model_rf_time.transform(tip_test)

print("logistic:", classification_report(pred_logreg_time))
print("forest  :", classification_report(pred_rf_time))

logistic: {'auc': 0.665, 'auc_pr': 0.804, 'accuracy': 0.766, 'f1': 0.713}
forest  : {'auc': 0.651, 'auc_pr': 0.796, 'accuracy': 0.759, 'f1': 0.689}


### Step 5: the confusion matrix

Rows are the true class, columns the prediction. `(1.0, 1.0)` are high tips correctly found, `(0.0, 0.0)` low tips correctly found, `(0.0, 1.0)` low tips wrongly called high, and `(1.0, 0.0)` high tips missed.

In [0]:
(pred_rf_time
    .groupBy("high_tip")
    .pivot("prediction", [0.0, 1.0])
    .count()
    .orderBy("high_tip")
    .show())

+--------+----+-----+
|high_tip| 0.0|  1.0|
+--------+----+-----+
|     0.0|2156|11582|
|     1.0|  87|34563|
+--------+----+-----+



**Reading the matrix (random forest, May).** The forest finds almost every high tip (34,563 of 34,650, 99.7%) but identifies only 2,156 of the 13,738 low tips (15.7%). It behaves close to "always predict high tip". Accuracy alone would hide this, so the next step looks at probabilities and thresholds.

### Step 6: choose a threshold instead of accepting 0.5

A model outputs a probability of "high tip"; the 0.5 cutoff is only a default. A higher threshold calls fewer trips "high tip": **precision** (how many of those calls are right) usually rises, **recall** (how many high tips are found) falls.

In [0]:
print("logistic regression, time split")
threshold_report(pred_logreg_time)

print("\nrandom forest, time split")
threshold_report(pred_rf_time)

logistic regression, time split
threshold 0.3: precision 0.754 | recall 0.996
threshold 0.4: precision 0.757 | recall 0.992
threshold 0.5: precision 0.760 | recall 0.984
threshold 0.6: precision 0.765 | recall 0.958
threshold 0.7: precision 0.777 | recall 0.838

random forest, time split
threshold 0.3: precision 0.716 | recall 1.000
threshold 0.4: precision 0.725 | recall 1.000
threshold 0.5: precision 0.749 | recall 0.997
threshold 0.6: precision 0.762 | recall 0.966
threshold 0.7: precision 0.767 | recall 0.892


**Reading the thresholds.** Across both models, precision stays between 0.72 and 0.78, close to the 0.716 positive rate in May. Raising the logistic threshold from 0.3 to 0.7 gains only about two points of precision (0.754 → 0.777) and loses 16 points of recall (0.996 → 0.838). The features separate high from low tippers only weakly, so no threshold gives a clearly useful operating point.

### Step 7: class weights

The classes are 72% / 28%. Giving each low-tip trip a weight of about 2.5 (the ratio of the two class sizes) makes both classes count equally in the loss. Weights change **where the model puts its threshold**, not how well it ranks trips, so AUC should stay about the same.

In [0]:
def add_class_weights(train_df, label="high_tip"):
    """Add a 'weight' column: the minority class gets n_majority / n_minority, the majority class gets 1.0."""
    n_pos = train_df.filter(F.col(label) == 1).count()
    n_neg = train_df.filter(F.col(label) == 0).count()

    minority = 1.0 if n_pos < n_neg else 0.0
    minority_weight = max(n_pos, n_neg) / min(n_pos, n_neg)

    return train_df.withColumn("weight", F.when(F.col(label) == minority, minority_weight).otherwise(1.0))


tip_train_w = add_class_weights(tip_train)

logreg_w = LogisticRegression(featuresCol="features", labelCol="high_tip", weightCol="weight",
                              regParam=0.01, maxIter=100)

model_logreg_w_time = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols, scale=True) + [logreg_w]
).fit(tip_train_w)

pred_logreg_w_time = model_logreg_w_time.transform(tip_test)

print("weighted logistic:", classification_report(pred_logreg_w_time))

weighted logistic: {'auc': 0.666, 'auc_pr': 0.805, 'accuracy': 0.688, 'f1': 0.688}


In [0]:
threshold_report(pred_logreg_w_time)

threshold 0.3: precision 0.761 | recall 0.981
threshold 0.4: precision 0.766 | recall 0.942
threshold 0.5: precision 0.782 | recall 0.783
threshold 0.6: precision 0.824 | recall 0.309
threshold 0.7: precision 0.850 | recall 0.013


**Reading the weighted model.** Class weights barely change the ranking quality (AUC 0.666 against 0.665 unweighted); they move the operating point. Accuracy falls (0.688 against 0.766) because the model now also tries to catch low tips, and at threshold 0.5 precision and recall are balanced (0.782 and 0.783). Accuracy is therefore not a fair way to compare these models, because the weights moved the operating point; compare AUC.

### Step 8: random split versus time split

The same two models and the weighted logistic are trained a second time on a **random 80/20 split**, where trips from all five months appear in both the training and the test set. Comparing both splits answers one question: **does a random split flatter the model?**

If the random split scores clearly higher, the model is learning month-specific patterns and the time split gives the honest estimate. If the scores are close, tipping behaviour is stable across January–May and the choice of split matters little here.

In [0]:
# Random 80/20 split of the same dataset: every month appears in both sets
tip_train_rand_raw, tip_test_rand_raw = tips.randomSplit([0.8, 0.2], seed=42)

# Freeze both halves as Delta tables so every re-run uses the same rows
tip_train_rand_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.tip_train_rand")
tip_test_rand_raw.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.tip_test_rand")

tip_train_rand = spark.table("taxi.taxi_ml.tip_train_rand")
tip_test_rand = spark.table("taxi.taxi_ml.tip_test_rand")

print(f"train {tip_train_rand.count():,} | test {tip_test_rand.count():,}")

train 172,114 | test 42,927


In [0]:
model_logreg_rand = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols, scale=True) + [logreg]
).fit(tip_train_rand)
model_rf_rand = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols) + [rf_clf]
).fit(tip_train_rand)

# Each model is scored on the test set of its own split
pred_logreg_rand = model_logreg_rand.transform(tip_test_rand)
pred_rf_rand = model_rf_rand.transform(tip_test_rand)

print("logistic:", classification_report(pred_logreg_rand))
print("forest  :", classification_report(pred_rf_rand))

logistic: {'auc': 0.665, 'auc_pr': 0.802, 'accuracy': 0.768, 'f1': 0.712}
forest  : {'auc': 0.648, 'auc_pr': 0.797, 'accuracy': 0.752, 'f1': 0.675}


In [0]:
(pred_rf_rand
    .groupBy("high_tip")
    .pivot("prediction", [0.0, 1.0])
    .count()
    .orderBy("high_tip")
    .show())

+--------+----+-----+
|high_tip| 0.0|  1.0|
+--------+----+-----+
|     0.0|1571|10623|
|     1.0|  19|30714|
+--------+----+-----+



In [0]:
print("logistic regression, random split")
threshold_report(pred_logreg_rand)

logistic regression, random split
threshold 0.3: precision 0.753 | recall 0.997
threshold 0.4: precision 0.756 | recall 0.994
threshold 0.5: precision 0.760 | recall 0.988
threshold 0.6: precision 0.763 | recall 0.969
threshold 0.7: precision 0.775 | recall 0.868


In [0]:
print("random forest, random split")
threshold_report(pred_rf_rand)

random forest, random split
threshold 0.3: precision 0.716 | recall 1.000
threshold 0.4: precision 0.721 | recall 1.000
threshold 0.5: precision 0.743 | recall 0.999
threshold 0.6: precision 0.756 | recall 0.991
threshold 0.7: precision 0.768 | recall 0.898


In [0]:
tip_train_rand_w = add_class_weights(tip_train_rand)

model_logreg_w_rand = Pipeline(
    stages=build_feature_stages(tip_cat_cols, tip_num_cols, scale=True) + [logreg_w]
).fit(tip_train_rand_w)

pred_logreg_w_rand = model_logreg_w_rand.transform(tip_test_rand)

print("weighted logistic:", classification_report(pred_logreg_w_rand))

weighted logistic: {'auc': 0.666, 'auc_pr': 0.803, 'accuracy': 0.704, 'f1': 0.698}


In [0]:
print("weighted logistic, random split")
threshold_report(pred_logreg_w_rand)

weighted logistic, random split
threshold 0.3: precision 0.760 | recall 0.986
threshold 0.4: precision 0.765 | recall 0.957
threshold 0.5: precision 0.780 | recall 0.816
threshold 0.6: precision 0.816 | recall 0.342
threshold 0.7: precision 0.829 | recall 0.017


In [0]:
def split_summary(name, train_df, test_df):
    """One row per set: size, number of months covered and share of high tips."""
    rows = []
    for part, df in [("train", train_df), ("test", test_df)]:
        stats = df.agg(
            F.count("*").alias("rows"),
            F.countDistinct("pickup_month").alias("months"),
            F.round(F.avg("high_tip"), 3).alias("high_tip_share"),
        ).first()
        rows.append((name, part, stats["rows"], stats["months"], stats["high_tip_share"]))
    return rows


composition = (split_summary("time split", tip_train, tip_test)
               + split_summary("random split", tip_train_rand, tip_test_rand))

spark.createDataFrame(composition, ["split", "set", "rows", "months", "high_tip_share"]).show(truncate=False)

+------------+-----+------+------+--------------+
|split       |set  |rows  |months|high_tip_share|
+------------+-----+------+------+--------------+
|time split  |train|166653|4     |0.717         |
|time split  |test |48388 |1     |0.716         |
|random split|train|172114|5     |0.717         |
|random split|test |42927 |5     |0.716         |
+------------+-----+------+------+--------------+



In [0]:
split_predictions = {
    ("time", "logistic"): pred_logreg_time,
    ("time", "forest"): pred_rf_time,
    ("time", "weighted logistic"): pred_logreg_w_time,
    ("random", "logistic"): pred_logreg_rand,
    ("random", "forest"): pred_rf_rand,
    ("random", "weighted logistic"): pred_logreg_w_rand,
}

comparison = spark.createDataFrame(
    [(split, model, *classification_report(pred).values()) for (split, model), pred in split_predictions.items()],
    ["split", "model", "auc", "auc_pr", "accuracy", "f1"],
)

comparison.orderBy("model", "split").show(truncate=False)

+------+-----------------+-----+------+--------+-----+
|split |model            |auc  |auc_pr|accuracy|f1   |
+------+-----------------+-----+------+--------+-----+
|random|forest           |0.648|0.797 |0.752   |0.675|
|time  |forest           |0.651|0.796 |0.759   |0.689|
|random|logistic         |0.665|0.802 |0.768   |0.712|
|time  |logistic         |0.665|0.804 |0.766   |0.713|
|random|weighted logistic|0.666|0.803 |0.704   |0.698|
|time  |weighted logistic|0.666|0.805 |0.688   |0.688|
+------+-----------------+-----+------+--------+-----+



### Step 8 (continued): the two splits compared

#### How the two splits differ

| | Time split | Random split |
|---|---|---|
| Training set | January–April | random 80% of all five months |
| Test set | May | random 20% of all five months |
| Months in each set | train: 4, test: 1 (disjoint) | all 5 in both |
| What it simulates | predicting a future month from the past | predicting trips similar to ones already seen |
| Main risk | May may differ from earlier months (drift, seasonality) | test trips have near-twins in the training set, so scores can be optimistic |
| Rows (train / test) | 166,653 / 48,388 | 172,114 / 42,927 |
| High-tip share (train / test) | 0.717 / 0.716 | 0.717 / 0.716 |

The class balance is the same in every set, so any difference in scores comes from *which trips* are in the sets, not from a changed class mix. The majority-class baseline is an accuracy of about 0.716 and an AUC of 0.5.

#### Results

| Model | Split | AUC | AUC-PR | Accuracy | F1 |
|---|---|---:|---:|---:|---:|
| Logistic regression | time | 0.665 | 0.804 | 0.766 | 0.713 |
| Logistic regression | random | 0.665 | 0.802 | 0.768 | 0.712 |
| Random forest | time | 0.651 | 0.796 | 0.759 | 0.689 |
| Random forest | random | 0.648 | 0.797 | 0.752 | 0.675 |
| Weighted logistic | time | 0.666 | 0.805 | 0.688 | 0.688 |
| Weighted logistic | random | 0.666 | 0.803 | 0.704 | 0.698 |

#### Interpretation

- **For logistic regression the two splits agree** to within 0.002 on every metric (AUC 0.665 in both).
- **The random split does not flatter the forest.** Its random-split AUC, accuracy and F1 are slightly *lower* (0.648 vs 0.651, 0.752 vs 0.759, 0.675 vs 0.689), and AUC-PR is the same within 0.001. A model exploiting near-identical trips shared between training and test would score higher, not lower.
- **The weighted logistic has the same AUC (0.666) in both splits.** Its accuracy and F1 are somewhat higher on the random split (0.704 vs 0.688 and 0.698 vs 0.688), but its threshold sits at a different point on each test set (recall at 0.5: 0.816 vs 0.783). Since the ranking quality is identical, this is not a better model.
- **Conclusion:** tipping behaviour looks stable from January to May, so the choice of split matters little here. The time split is still the number to report, because it is the only one that mimics real use (training on the past and predicting the future), and it would reveal drift if more months were added.

## Project B: tuning and validation

So far the Project B models use hand-set settings and a fixed 0.5 threshold. This section chooses the settings with cross-validation on the training months only, picks a decision threshold, checks whether the scores are honest probabilities, and scores May once.

### Step 1: month folds and class weights

Train on January to April and keep May untouched as the final test. Each training month gets a fold number (0 to 3), so cross-validation holds out one month at a time. Class weights balance the classes during training only; evaluation stays unweighted so the scores reflect the real class mix. The high-tip class gets the weight negatives ÷ positives (0.39), which gives the same class balance as the weight of about 2.5 on low tips used earlier in this project.

**Metric choice.** `areaUnderROC` is the tuning metric (Step 2 explains why). Both AUC-ROC and AUC-PR are reported on the final test.

In [0]:
# One fold per training month: each month takes a turn as the validation set
months_b = ["2026-01", "2026-02", "2026-03", "2026-04"]
fold_b = F.coalesce(*[F.when(F.col("pickup_month") == m, F.lit(i)) for i, m in enumerate(months_b)])

train_b = tips.filter(F.col("pickup_month") <= "2026-04").withColumn("fold", fold_b)
test_b  = tips.filter(F.col("pickup_month") == "2026-05")

# Weight the positive class by (negatives / positives): both classes then carry equal total weight
pos_share = train_b.agg(F.avg("high_tip")).first()[0]
w_pos = (1 - pos_share) / pos_share
train_b = train_b.withColumn("weight", F.when(F.col("high_tip") == 1.0, F.lit(w_pos)).otherwise(1.0))

print(f"train rows: {train_b.count():,} | test rows (May): {test_b.count():,}")
print(f"positive share in train: {pos_share:.3f} -> positive weight {w_pos:.2f}")

# Is the balance stable across months? (it affects how far to trust one validation month)
(tips.groupBy("pickup_month")
     .agg(F.count("*").alias("trips"), F.round(F.avg("high_tip"), 3).alias("positive_share"))
     .orderBy("pickup_month").show())

train rows: 166,653 | test rows (May): 48,388
positive share in train: 0.717 -> positive weight 0.39
+------------+-----+--------------+
|pickup_month|trips|positive_share|
+------------+-----+--------------+
|     2026-01|39026|         0.719|
|     2026-02|36081|         0.715|
|     2026-03|45324|         0.715|
|     2026-04|46222|         0.718|
|     2026-05|48388|         0.716|
+------------+-----+--------------+



### Step 2: tuning logistic regression with month folds

The positive class (high tip) is the majority at 71.7%, stable across months (0.715 to 0.719). A model with no skill scores 0.5 on `areaUnderROC` but about 0.72 on `areaUnderPR`, so `areaUnderROC` is used as the tuning metric; both are reported on the final test.

`regParam` is the strength of the penalty on large coefficients, and `elasticNetParam` mixes the two penalty types (0 is ridge, 1 is lasso). The grid has 3 x 2 = 6 settings, each trained 4 times (one per held-out month), plus one final refit.

In [0]:
metric = "areaUnderROC"

logreg_tune = LogisticRegression(featuresCol="features", labelCol="high_tip",
                                 weightCol="weight", maxIter=100)
pipe_lr = Pipeline(stages=build_feature_stages(tip_cat_cols, tip_num_cols, scale=True) + [logreg_tune])

grid_lr = (ParamGridBuilder()
           .addGrid(logreg_tune.regParam, [0.001, 0.01, 0.1])
           .addGrid(logreg_tune.elasticNetParam, [0.0, 0.5])
           .build())

cv_lr = CrossValidator(estimator=pipe_lr, estimatorParamMaps=grid_lr,
                       evaluator=BinaryClassificationEvaluator(labelCol="high_tip", metricName=metric),
                       numFolds=4, foldCol="fold", seed=42, parallelism=2)

print(f"fits: {len(grid_lr) * 4 + 1}")
cv_lr_model = cv_lr.fit(train_b)

stds = getattr(cv_lr_model, "stdMetrics", None)      # may not exist on serverless
rows = []
for i, (params, avg) in enumerate(zip(grid_lr, cv_lr_model.avgMetrics)):
    row = {p.name: v for p, v in params.items()}
    row[f"cv_{metric}"] = round(avg, 4)
    if stds is not None:
        row["cv_std"] = round(stds[i], 4)
    rows.append(row)

results_lr = pd.DataFrame(rows).sort_values(f"cv_{metric}", ascending=False).reset_index(drop=True)
display(results_lr)

best_lr = cv_lr_model.bestModel.stages[-1]
print("best regParam:", best_lr.getOrDefault("regParam"),
      "| best elasticNetParam:", best_lr.getOrDefault("elasticNetParam"))

fits: 25


regParam,elasticNetParam,cv_areaUnderROC,cv_std
0.001,0.0,0.6662,0.0055
0.01,0.0,0.666,0.0054
0.001,0.5,0.6659,0.0054
0.1,0.0,0.6626,0.0054
0.01,0.5,0.6541,0.0058
0.1,0.5,0.6351,0.0041


best regParam: 0.001 | best elasticNetParam: 0.0


**Reading Step 2.** The best logistic regression (`regParam=0.001`, `elasticNetParam=0`) reaches a cross-validated AUC of 0.666, with a spread of about 0.006 across the four held-out months. The four ridge-type settings are tied within that spread, so regularisation hardly matters at this data size; only a strong lasso penalty (`regParam=0.1`, `elasticNetParam=0.5`, AUC 0.635) clearly hurts, probably because it removes useful features (not checked). An AUC of 0.666 against the no-skill 0.5 shows modest signal. 
A plausible reason, not tested here: whether a rider tips 20% or more depends mostly on the rider, which the trip data does not contain.

### Step 3: tuning the gradient-boosted classifier

Trees need no scaling, and zones go in as category indices (`maxBins=300`, because there are about 256 zones). The full grid would be 8 settings x 4 folds, so it is halved to `maxDepth` in {3, 5} and `maxIter` in {20, 40} with `stepSize` fixed at 0.1: 4 settings x 4 folds = 16 fits plus one refit. The search runs on a 50% sample of the January to April rows, and the winning settings are refitted on all of `train_b` in Step 6.

In [0]:
gbt_c = GBTClassifier(featuresCol="features", labelCol="high_tip", weightCol="weight",
                      stepSize=0.1, maxBins=300, seed=42)

pipe_gbt = Pipeline(stages=build_tree_stages(tip_cat_cols, tip_num_cols) + [gbt_c])

grid_gbt = (ParamGridBuilder()
            .addGrid(gbt_c.maxDepth, [3, 5])
            .addGrid(gbt_c.maxIter, [20, 40])
            .build())                                   # 2 x 2 = 4 settings

cv_gbt = CrossValidator(estimator=pipe_gbt, estimatorParamMaps=grid_gbt,
                        evaluator=BinaryClassificationEvaluator(labelCol="high_tip", metricName=metric),
                        numFolds=4, foldCol="fold", seed=42, parallelism=2)

search_df = train_b.sample(fraction=0.5, seed=1)        # note: the fold column is carried along
print(f"search rows: {search_df.count():,} | fits: {len(grid_gbt) * 4 + 1}")
cv_gbt_model = cv_gbt.fit(search_df)

stds = getattr(cv_gbt_model, "stdMetrics", None)
rows = []
for i, (params, avg) in enumerate(zip(grid_gbt, cv_gbt_model.avgMetrics)):
    row = {p.name: v for p, v in params.items()}
    row[f"cv_{metric}"] = round(avg, 4)
    if stds is not None:
        row["cv_std"] = round(stds[i], 4)
    rows.append(row)

results_gbt = pd.DataFrame(rows).sort_values(f"cv_{metric}", ascending=False).reset_index(drop=True)
display(results_gbt)

best_gbt_clf = cv_gbt_model.bestModel.stages[-1]
best_gbt_params = {p: best_gbt_clf.getOrDefault(p) for p in ["maxDepth", "maxIter", "stepSize"]}

print("best settings:", best_gbt_params)

search rows: 82,892 | fits: 17


maxDepth,maxIter,cv_areaUnderROC,cv_std
3,40,0.6729,0.0043
3,20,0.672,0.004
5,20,0.6636,0.005
5,40,0.6604,0.0051


best settings: {'maxDepth': 3, 'maxIter': 40, 'stepSize': 0.1}


### Step 3 (continued): extending the grid downward

The first GBT search put its best depth at the lowest value tried (`maxDepth=3`), and more rounds did not help at depth 5. Shallower trees usually need more rounds, so the grid is extended to `maxDepth` in {2, 3} and `maxIter` in {40, 80}: 4 settings x 4 folds = 16 fits plus one refit, on the same 50% sample as before.

In [0]:
grid_gbt2 = (ParamGridBuilder()
             .addGrid(gbt_c.maxDepth, [2, 3])
             .addGrid(gbt_c.maxIter, [40, 80])
             .build())

cv_gbt2 = CrossValidator(estimator=pipe_gbt, estimatorParamMaps=grid_gbt2,
                         evaluator=BinaryClassificationEvaluator(labelCol="high_tip", metricName=metric),
                         numFolds=4, foldCol="fold", seed=42, parallelism=2)

print(f"fits: {len(grid_gbt2) * 4 + 1}")
cv_gbt_model2 = cv_gbt2.fit(search_df)

stds2 = getattr(cv_gbt_model2, "stdMetrics", None)
rows2 = []
for i, (params, avg) in enumerate(zip(grid_gbt2, cv_gbt_model2.avgMetrics)):
    row = {p.name: v for p, v in params.items()}
    row[f"cv_{metric}"] = round(avg, 4)
    if stds2 is not None:
        row["cv_std"] = round(stds2[i], 4)
    rows2.append(row)
results_gbt2 = pd.DataFrame(rows2)

# Both searches together (the depth 3 / 40 rounds setting appears in both; keep the second run)
results_all = (pd.concat([results_gbt, results_gbt2])
                 .drop_duplicates(subset=["maxDepth", "maxIter"], keep="last")
                 .sort_values(f"cv_{metric}", ascending=False)
                 .reset_index(drop=True))
display(results_all)

best_row = results_all.iloc[0]
best_gbt_params = {"maxDepth": int(best_row["maxDepth"]), "maxIter": int(best_row["maxIter"]), "stepSize": 0.1}
print("best settings so far:", best_gbt_params)

fits: 17


maxDepth,maxIter,cv_areaUnderROC,cv_std
2,80,0.6746,0.0052
2,40,0.6735,0.0049
3,40,0.6729,0.0043
3,20,0.672,0.004
3,80,0.6706,0.0046
5,20,0.6636,0.005
5,40,0.6604,0.0051


best settings so far: {'maxDepth': 2, 'maxIter': 80, 'stepSize': 0.1}


**Reading Step 3 (continued).** Depth matters more than rounds. Depth-5 trees overfit (CV AUC 0.660 to 0.664), while depth 2 and depth 3 are tied within the fold-to-fold spread of about 0.005 (0.6706 to 0.6746). Extending the grid downward showed the curve flat between depth 2 and depth 3. Depth 2 is still the lowest depth tried (depth 1 was not tested), but the two depths are tied, so going lower is unlikely to help much. By the one-standard-error rule the simplest setting inside the tie is `maxDepth=2, maxIter=40` (CV AUC 0.6735, against 0.6746 for the best raw score). That depth-2 trees are enough, and that the additive logistic regression (0.666) is close behind, suggests the tipping signal in these features is weak and mostly additive: interactions between trip features add little.

### Step 4: choosing the decision threshold on April

The model is trained on January to March with the settings chosen above (`maxDepth=2`, `maxIter=40`, `stepSize=0.1`), then scores April, which it has not seen. April was one of the four cross-validation folds, so it influenced the choice of settings; here it is used for a different decision (the threshold), and May stays untouched for the final test.

The rule is fixed before looking at the table: choose the threshold that maximises **balanced accuracy**, the average of recall on high tips and specificity on non-high tips. F1 for the positive class is not used because 71.7% of trips are positive, so predicting "high tip" for everyone already scores well on it.

In [0]:
best_gbt_params = {"maxDepth": 2, "maxIter": 40, "stepSize": 0.1}     # one-standard-error choice

est = GBTClassifier(featuresCol="features", labelCol="high_tip", weightCol="weight",
                    maxBins=300, seed=42, **best_gbt_params)

fit_m = Pipeline(stages=build_tree_stages(tip_cat_cols, tip_num_cols) + [est]).fit(
    train_b.filter(F.col("pickup_month") <= "2026-03"))

val_scored = (fit_m.transform(train_b.filter(F.col("pickup_month") == "2026-04"))
              .withColumn("p", vector_to_array("probability")[1]))

print("April score range:", val_scored.agg(F.min("p"), F.max("p")).first())

thresholds = [round(0.1 + 0.05 * i, 2) for i in range(17)]            # 0.10 ... 0.90
pos = F.col("high_tip") == 1.0

aggs = []
for t in thresholds:
    k = int(round(t * 100))
    hit = F.col("p") >= t
    aggs += [F.sum(F.when(hit & pos, 1).otherwise(0)).alias(f"tp_{k}"),
             F.sum(F.when(hit & ~pos, 1).otherwise(0)).alias(f"fp_{k}"),
             F.sum(F.when(~hit & pos, 1).otherwise(0)).alias(f"fn_{k}"),
             F.sum(F.when(~hit & ~pos, 1).otherwise(0)).alias(f"tn_{k}")]
counts = val_scored.agg(*aggs).first().asDict()          # one pass over the data for all thresholds

rows = []
for t in thresholds:
    k = int(round(t * 100))
    tp, fp, fn, tn = counts[f"tp_{k}"], counts[f"fp_{k}"], counts[f"fn_{k}"], counts[f"tn_{k}"]
    recall = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    precision = tp / (tp + fp) if tp + fp else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if tp > 0 else 0.0
    rows.append({"threshold": t,
                 "recall": round(recall, 3), "specificity": round(specificity, 3),
                 "balanced_acc": round((recall + specificity) / 2, 4),
                 "precision": round(precision, 3), "f1": round(f1, 3),
                 "share_flagged": round((tp + fp) / (tp + fp + fn + tn), 3)})

thr = pd.DataFrame(rows)
display(thr)

best_threshold = float(thr.loc[thr["balanced_acc"].idxmax(), "threshold"])
print("threshold that maximises balanced accuracy on April:", best_threshold)

April score range: Row(min(p)=0.022510301603402838, max(p)=0.8156303004560825)


threshold,recall,specificity,balanced_acc,precision,f1,share_flagged
0.1,0.998,0.177,0.5877,0.756,0.86,0.949
0.15,0.995,0.2,0.5972,0.76,0.862,0.94
0.2,0.991,0.212,0.6015,0.762,0.862,0.934
0.25,0.988,0.22,0.6041,0.764,0.861,0.929
0.3,0.983,0.231,0.6069,0.765,0.861,0.923
0.35,0.976,0.242,0.6091,0.767,0.859,0.915
0.4,0.964,0.259,0.6117,0.768,0.855,0.901
0.45,0.922,0.315,0.6183,0.774,0.842,0.855
0.5,0.802,0.452,0.627,0.789,0.795,0.731
0.55,0.578,0.656,0.6166,0.811,0.674,0.512


threshold that maximises balanced accuracy on April: 0.5


**Reading Step 4.** Balanced accuracy peaks at a threshold of 0.50 (0.627), ahead of 0.45 (0.618) and 0.55 (0.617), so the threshold is a clear peak and not a plateau. At 0.50 the model flags 73% of April trips, finds 80% of the real high tippers (recall 0.802) and correctly clears 45% of the others (specificity 0.452); precision is 0.789 against a base rate of 0.717. F1 would have chosen a threshold of 0.15 to 0.20 where over 93% of trips are flagged, barely better than flagging everything (F1 about 0.835), which is why balanced accuracy was chosen as the rule. Thresholds from 0.70 up flag under 1% of trips and are not meaningful. The model's scores span only 0.02 to 0.82, consistent with the modest AUC.

### Step 5: are the probabilities honest? A calibration table

April's scores are cut into ten bins of equal size (deciles of the predicted probability), and the average prediction in each bin is compared with the actual share of high tips. Two things are checked: whether the actual rate rises from the lowest decile to the highest (the model ranks trips correctly), and whether the predicted values match the actual rates (calibration). Because training weights downweight the majority class (high tips), the predicted values are expected to run below the actual rates.

In [0]:
cal = (val_scored
       .withColumn("decile", F.ntile(10).over(Window.orderBy("p")))
       .groupBy("decile")
       .agg(F.count("*").alias("trips"),
            F.round(F.min("p"), 3).alias("min_p"),
            F.round(F.max("p"), 3).alias("max_p"),
            F.round(F.avg("p"), 3).alias("avg_predicted"),
            F.round(F.avg("high_tip"), 3).alias("actual_rate"))
       .withColumn("gap", F.round(F.col("avg_predicted") - F.col("actual_rate"), 3))
       .orderBy("decile"))
display(cal)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


decile,trips,min_p,max_p,avg_predicted,actual_rate,gap
1,4623,0.023,0.402,0.156,0.266,-0.11
2,4623,0.402,0.477,0.45,0.674,-0.224
3,4622,0.477,0.509,0.494,0.697,-0.203
4,4622,0.509,0.531,0.521,0.733,-0.212
5,4622,0.531,0.552,0.542,0.753,-0.211
6,4622,0.552,0.573,0.562,0.776,-0.214
7,4622,0.573,0.597,0.584,0.784,-0.2
8,4622,0.597,0.627,0.612,0.816,-0.204
9,4622,0.627,0.655,0.641,0.832,-0.191
10,4622,0.655,0.816,0.677,0.852,-0.175


**Reading Step 5.** The actual high-tip rate rises in every decile of the score, from 0.266 in the lowest to 0.852 in the highest, so the model ranks trips correctly; most of its separation comes from the lowest decile (trips that tip little). The predicted values run about 0.2 below the actual rates in deciles 2 to 9. This is a direct effect of the class weights: the positive weight of 0.39 makes the model behave as if high tips were as common as other trips, which is also why the best threshold is 0.5 (a score of 0.5 corresponds to a real rate close to the 0.717 base rate). The shift has a known form, and rescaling with `p / (p + w_pos * (1 - p))` brings the predicted values back to the actual rates (see the check below). The scores are therefore read as ranks and a threshold, not as probabilities, unless they are rescaled.

In [0]:
val_cal = val_scored.withColumn(
    "p_cal", F.col("p") / (F.col("p") + F.lit(w_pos) * (1 - F.col("p"))))

cal2 = (val_cal
        .withColumn("decile", F.ntile(10).over(Window.orderBy("p_cal")))
        .groupBy("decile")
        .agg(F.count("*").alias("trips"),
             F.round(F.avg("p_cal"), 3).alias("avg_rescaled"),
             F.round(F.avg("high_tip"), 3).alias("actual_rate"))
        .withColumn("gap", F.round(F.col("avg_rescaled") - F.col("actual_rate"), 3))
        .orderBy("decile"))
display(cal2)

val_cal.agg(
    F.round(F.avg((F.col("p") - F.col("high_tip")) ** 2), 4).alias("brier_raw"),
    F.round(F.avg((F.col("p_cal") - F.col("high_tip")) ** 2), 4).alias("brier_rescaled"),
).show()

decile,trips,avg_rescaled,actual_rate,gap
1,4623,0.287,0.266,0.021
2,4623,0.674,0.674,0.0
3,4622,0.712,0.697,0.015
4,4622,0.734,0.733,0.001
5,4622,0.75,0.753,-0.003
6,4622,0.765,0.776,-0.011
7,4622,0.781,0.784,-0.003
8,4622,0.8,0.816,-0.016
9,4622,0.819,0.832,-0.013
10,4622,0.841,0.852,-0.011


+---------+--------------+
|brier_raw|brier_rescaled|
+---------+--------------+
|   0.2103|        0.1702|
+---------+--------------+



**Reading the rescaling check.** After rescaling with `p / (p + w_pos * (1 - p))`, the predicted values match the actual April rates within 0.021 in every decile (most within 0.016), against gaps of 0.11 to 0.22 before. The correction uses only the training weight, nothing fitted on April, so this is an out-of-sample check. The Brier score falls from 0.2103 to 0.1702; always predicting the base rate would score about 0.203, so the raw weighted scores were slightly worse than that and the rescaled ones are about 16% better. A small pattern remains (slightly high in the lowest decile, slightly low in the top five), meaning the model's spread is a little too narrow, which is expected for shallow boosted trees. With the rescaling, the scores can be read as probabilities; the ranking, and therefore the AUC and the threshold decision, are unchanged.

### Step 6: the final fit and the one test

The chosen settings (`maxDepth=2`, `maxIter=40`, `stepSize=0.1`) are refitted on all of January to April and scored once on May, with the threshold of 0.50 chosen on April. The tuned logistic regression (refitted on all of `train_b` by the cross-validation) and a majority-class baseline are scored on the same rows. May is not used for any further decision after this cell.

In [0]:
est_final = GBTClassifier(featuresCol="features", labelCol="high_tip", weightCol="weight",
                          maxBins=300, seed=42, **best_gbt_params)
final_b = Pipeline(stages=build_tree_stages(tip_cat_cols, tip_num_cols) + [est_final]).fit(train_b)

test_gbt = final_b.transform(test_b).withColumn("p", vector_to_array("probability")[1])
test_lr  = cv_lr_model.bestModel.transform(test_b)           # already refitted on all of train_b

may_share = test_b.agg(F.avg("high_tip")).first()[0]

def auc(pred_df, m):
    return BinaryClassificationEvaluator(labelCol="high_tip", metricName=m).evaluate(pred_df)

results_b = pd.DataFrame([
    {"model": "majority class (always high tip)", "auc_roc": 0.5, "auc_pr": round(may_share, 4)},
    {"model": "logistic regression, tuned", "auc_roc": round(auc(test_lr, "areaUnderROC"), 4),
     "auc_pr": round(auc(test_lr, "areaUnderPR"), 4)},
    {"model": "GBT classifier, tuned", "auc_roc": round(auc(test_gbt, "areaUnderROC"), 4),
     "auc_pr": round(auc(test_gbt, "areaUnderPR"), 4)},
])
print(f"May positive share (no-skill AUC-PR): {may_share:.3f}")
display(results_b)

# Decision quality at the April-chosen threshold
pos = F.col("high_tip") == 1.0
hit = F.col("p") >= best_threshold
c = test_gbt.agg(
    F.sum(F.when(hit & pos, 1).otherwise(0)).alias("tp"),
    F.sum(F.when(hit & ~pos, 1).otherwise(0)).alias("fp"),
    F.sum(F.when(~hit & pos, 1).otherwise(0)).alias("fn"),
    F.sum(F.when(~hit & ~pos, 1).otherwise(0)).alias("tn")).first()

recall = c.tp / (c.tp + c.fn)
specificity = c.tn / (c.tn + c.fp)
precision = c.tp / (c.tp + c.fp)
print(f"threshold {best_threshold}: recall {recall:.3f} | specificity {specificity:.3f} | "
      f"balanced accuracy {(recall + specificity) / 2:.4f} | precision {precision:.3f} | "
      f"flagged {(c.tp + c.fp) / (c.tp + c.fp + c.fn + c.tn):.3f}")
print(f"confusion: TP {c.tp:,} | FP {c.fp:,} | FN {c.fn:,} | TN {c.tn:,}")

# Probabilities after the weight correction
test_cal = test_gbt.withColumn("p_cal", F.col("p") / (F.col("p") + F.lit(w_pos) * (1 - F.col("p"))))
test_cal.agg(
    F.round(F.avg((F.col("p") - F.col("high_tip")) ** 2), 4).alias("brier_raw"),
    F.round(F.avg((F.col("p_cal") - F.col("high_tip")) ** 2), 4).alias("brier_rescaled"),
    F.round(F.avg("p_cal"), 3).alias("avg_rescaled"),
    F.round(F.avg("high_tip"), 3).alias("actual_rate")).show()

May positive share (no-skill AUC-PR): 0.716


model,auc_roc,auc_pr
majority class (always high tip),0.5,0.7161
"logistic regression, tuned",0.666,0.8049
"GBT classifier, tuned",0.6704,0.8094


threshold 0.5: recall 0.730 | specificity 0.504 | balanced accuracy 0.6170 | precision 0.788 | flagged 0.663
confusion: TP 25,288 | FP 6,812 | FN 9,362 | TN 6,926
+---------+--------------+------------+-----------+
|brier_raw|brier_rescaled|avg_rescaled|actual_rate|
+---------+--------------+------------+-----------+
|   0.2129|        0.1732|       0.717|      0.716|
+---------+--------------+------------+-----------+



**Reading Step 6.** On May, which was untouched until this cell:

| Model | Settings | CV AUC-ROC (4 month folds) | May AUC-ROC | May AUC-PR |
| --- | --- | --- | --- | --- |
| Majority class (always high tip) | none | - | 0.500 | 0.716 |
| Logistic regression, tuned | `regParam=0.001`, `elasticNetParam=0` | 0.666 (spread 0.006) | 0.666 | 0.805 |
| GBT classifier, tuned | `maxDepth=2`, `maxIter=40`, `stepSize=0.1` | 0.674 (spread 0.005, on a 50% sample) | 0.670 | 0.809 |

- **Test performance:** the GBT's lead of about 0.004 AUC-ROC is one fold-to-fold spread, so the tuned GBT and the tuned logistic regression are effectively tied; both clearly beat the no-skill levels of 0.5 and 0.716 (the May positive share).
- **Validation versus test:** cross-validation was a reliable guide: it predicted 0.674 (GBT, searched on half the data) and 0.666 (logistic regression), and May gave 0.670 and 0.666.
- **Decision at the April threshold of 0.50:** the GBT flags 66% of May trips, finds 73% of the real high tippers (recall 0.730) and clears 50% of the others (specificity 0.504); balanced accuracy is 0.617 (0.627 on April) and precision 0.788 against a base rate of 0.716.
- **Probabilities:** after the weight correction the mean predicted probability is 0.717 against an actual 0.716, and the Brier score is 0.173 against 0.203 for always predicting the base rate, so the rescaled scores can be read as probabilities.
- **Overall:** the features carry real but limited information about whether a rider tips 20% or more, and tuning the model cannot add what the features do not contain.

### Step 7: random split versus time-based split

Step 8 of the first part of Project B made this comparison for the hand-set models, where the AUCs agreed within 0.003. Here it is repeated for the tuned GBT, with three random seeds to show the chance variation.

The final GBT settings are refitted on random 80/20 splits of all five months (three seeds) and scored on the held-out 20%, then compared with the time-based result (train January to April, test May). A random split lets the model see trips from the same weeks it is tested on, so it can look slightly better than the time-based split, which mimics real use (the model has never seen the future month). The three seeds show how much the random-split score varies by chance alone, which is the yardstick for reading the gap.

In [0]:
tips_w = tips.withColumn("weight", F.when(F.col("high_tip") == 1.0, F.lit(w_pos)).otherwise(1.0))

def fit_score(train_df, test_df):
    est = GBTClassifier(featuresCol="features", labelCol="high_tip", weightCol="weight",
                        maxBins=300, seed=42, **best_gbt_params)
    m = Pipeline(stages=build_tree_stages(tip_cat_cols, tip_num_cols) + [est]).fit(train_df)
    pred = m.transform(test_df)
    return {"train_rows": train_df.count(), "test_rows": test_df.count(),
            "auc_roc": round(auc(pred, "areaUnderROC"), 4),
            "auc_pr": round(auc(pred, "areaUnderPR"), 4)}

rows = [{"split": "time-based (train Jan-Apr, test May)",
         "train_rows": train_b.count(), "test_rows": test_b.count(),
         "auc_roc": round(auc(test_gbt, "areaUnderROC"), 4),
         "auc_pr": round(auc(test_gbt, "areaUnderPR"), 4)}]

for seed in [1, 2, 3]:
    r = F.rand(seed)          # same seed on the same table: the train and test parts do not overlap
    rows.append({"split": f"random 80/20, seed {seed}",
                 **fit_score(tips_w.filter(r < 0.8), tips_w.filter(r >= 0.8))})

split_cmp = pd.DataFrame(rows)
display(split_cmp)

rand_rows = split_cmp[split_cmp["split"].str.startswith("random")]
print(f"random-split AUC-ROC: mean {rand_rows['auc_roc'].mean():.4f}, "
      f"range {rand_rows['auc_roc'].min():.4f} to {rand_rows['auc_roc'].max():.4f}")

split,train_rows,test_rows,auc_roc,auc_pr
"time-based (train Jan-Apr, test May)",166653,48388,0.6704,0.8094
"random 80/20, seed 1",171758,43283,0.6747,0.8106
"random 80/20, seed 2",171941,43100,0.6778,0.8143
"random 80/20, seed 3",171912,43129,0.6775,0.8147


random-split AUC-ROC: mean 0.6767, range 0.6747 to 0.6778


**Reading Step 7.**

- **Result:** AUC-ROC 0.6704 on the time-based split (test May) against 0.6747, 0.6778 and 0.6775 on three random 80/20 splits (mean 0.6767); AUC-PR shows the same pattern (0.8094 against a mean of 0.8132).
- **Direction:** all three random seeds beat the time-based result, by 0.004 to 0.007 AUC-ROC, so a random split is slightly optimistic, as expected: it lets the model train on trips from the same weeks it is tested on, while the time-based split tests on a month the model has never seen.
- **Size:** the gap is about twice the range across seeds (0.003) but small in absolute terms, comparable to the fold-to-fold spread in cross-validation, so month-to-month drift is limited.
- **Caveats:** only three seeds were run, and the test sets differ (May only against a mix of all months), so part of the gap may be May being slightly harder, not the effect of the split.
- **Decision:** the time-based figure is the one to quote, because it matches how the model would be used.

### Step 8: save the tuned model and track the runs

The same routine as Project A: save the fitted `PipelineModel`, prove that a reloaded copy gives the same predictions, and log every model as an MLflow run, now with `task=tip` so these runs stay separate from the fare runs. One extra step: the model is not usable without its **decision settings**. The scores are class-weighted, so the threshold of 0.50 applies to the raw score, and a real probability needs the rescaling `p / (p + w_pos * (1 - p))`. Both go into a small JSON file inside the model folder, and into the run parameters.

In [0]:
TIP_MODEL_DIR = "/Volumes/taxi/taxi_ml/ml_models/tip_gbt_tuned"

final_b.write().overwrite().save(TIP_MODEL_DIR)
reloaded_b = PipelineModel.load(TIP_MODEL_DIR)

# The model needs its decision settings to be used correctly: keep them next to it
decision_settings = {
    "threshold_on_raw_score": best_threshold,
    "positive_class_weight": round(w_pos, 4),
    "rescaling_to_probability": "p_cal = p / (p + w_pos * (1 - p))",
    "chosen_on": "April 2026 (balanced accuracy)",
}
with open(f"{TIP_MODEL_DIR}/decision_settings.json", "w") as f:
    json.dump(decision_settings, f, indent=2)

# The same 500 raw May trips go through both models
check_df = spark.createDataFrame(test_b.limit(500).toPandas())

def p_high(model, df):
    return model.transform(df).select(vector_to_array("probability")[1].alias("p")).toPandas()["p"]

max_diff = float((p_high(final_b, check_df) - p_high(reloaded_b, check_df)).abs().max())
size_mb = sum(os.path.getsize(os.path.join(root, f))
              for root, _, files in os.walk(TIP_MODEL_DIR) for f in files) / 1e6

print(f"largest probability difference: {max_diff:.2e} | model folder: {size_mb:.2f} MB")

# Full May check: the reloaded model must reproduce the tuned model's scores exactly
print("in memory:", classification_report(test_gbt))
print("reloaded :", classification_report(reloaded_b.transform(test_b)))

largest probability difference: 0.00e+00 | model folder: 0.04 MB
in memory: {'auc': 0.67, 'auc_pr': 0.809, 'accuracy': 0.666, 'f1': 0.674}
reloaded : {'auc': 0.67, 'auc_pr': 0.809, 'accuracy': 0.666, 'f1': 0.674}


In [0]:
manifest_b = write_manifest(
    TIP_MODEL_DIR, "tip_gbt_tuned",
    tables=["taxi.taxi_ml.tip_dataset"],
    feature_df=train_b, categorical=tip_cat_cols, numeric=tip_num_cols,
    split={"method": "time-based", "train": "2026-01 to 2026-04", "test": "2026-05",
           "train_rows": train_b.count(), "test_rows": test_b.count()},
    hyperparameters={"model": "GBTClassifier", **best_gbt_params, "maxBins": 300,
                     "positive_class_weight": round(w_pos, 4), "cv_folds": 4},
    metrics=classification_report(test_gbt),
    extra={"decision_settings": decision_settings},
)
print(json.dumps({k: manifest_b[k] for k in ["input_tables", "split", "metrics"]}, indent=2))

{
  "input_tables": [
    {
      "table": "taxi.taxi_ml.tip_dataset",
      "delta_version": 24,
      "last_write": "2026-11-28 09:11:58",
      "rows": 215041
    }
  ],
  "split": {
    "method": "time-based",
    "train": "2026-01 to 2026-04",
    "test": "2026-05",
    "train_rows": 166653,
    "test_rows": 48388
  },
  "metrics": {
    "auc": 0.67,
    "auc_pr": 0.809,
    "accuracy": 0.666,
    "f1": 0.674
  }
}


In [0]:
TAGS_TIP = {"task": "tip", "split": "time-based: train Jan-Apr, test May", "seed": "42",
            "sample": "card trips of the 2% sample (tip_dataset)"}

# Cross-validation scores of the chosen settings (from the tuning tables above)
cv_auc_lr = float(results_lr.iloc[0]["cv_areaUnderROC"])
gbt_row = results_all[(results_all["maxDepth"] == 2) & (results_all["maxIter"] == 40)].iloc[0]
cv_auc_gbt = float(gbt_row["cv_areaUnderROC"])
best_lr = cv_lr_model.bestModel.stages[-1]

# Majority class: no model, so the scores are known by definition
log_run("baseline_majority_class",
        {"model": "constant: always predict high tip"},
        {"auc": 0.5, "auc_pr": round(may_share, 3), "accuracy": round(may_share, 3)},
        "none", tags=TAGS_TIP)

# The three hand-set models from the first part of Project B (time split, scored on May)
hand_set = [
    ("logistic_regression", pred_logreg_time,
     {"model": "LogisticRegression", "regParam": logreg.getRegParam(), "maxIter": logreg.getMaxIter(),
      "class_weights": "none"}),
    ("random_forest", pred_rf_time,
     {"model": "RandomForestClassifier", "numTrees": rf_clf.getNumTrees(), "maxDepth": rf_clf.getMaxDepth(),
      "class_weights": "none"}),
    ("logistic_regression_class_weights", pred_logreg_w_time,
     {"model": "LogisticRegression", "regParam": logreg_w.getRegParam(), "maxIter": logreg_w.getMaxIter(),
      "class_weights": "low tips x (n_high / n_low), about 2.5"}),
]
for run_name, pred_df, params in hand_set:
    log_run(run_name, params, classification_report(pred_df), "B", tags=TAGS_TIP)

# The tuned logistic regression (refitted on all of train_b by the cross-validation)
log_run("logistic_regression_tuned",
        {"model": "LogisticRegression", "regParam": best_lr.getOrDefault("regParam"),
         "elasticNetParam": best_lr.getOrDefault("elasticNetParam"), "maxIter": best_lr.getOrDefault("maxIter"),
         "class_weights": f"high tips x {w_pos:.2f}", "cv_folds": 4, "cv_auc_roc": round(cv_auc_lr, 4)},
        classification_report(test_lr), "B", tags=TAGS_TIP, extra_tags={"stage": "tuned"})

# The tuned GBT: settings, how it was chosen, decision quality on May, and the saved model files
brier = test_cal.agg(F.avg((F.col("p_cal") - F.col("high_tip")) ** 2)).first()[0]

log_run("gbt_tuned",
        {"model": "GBTClassifier", "maxDepth": best_gbt_params["maxDepth"], "maxIter": best_gbt_params["maxIter"],
         "stepSize": best_gbt_params["stepSize"], "maxBins": 300,
         "class_weights": f"high tips x {w_pos:.2f}", "threshold_on_raw_score": best_threshold,
         "cv_folds": 4, "cv_search_sample": "50% of Jan-Apr", "cv_auc_roc": round(cv_auc_gbt, 4)},
        {**classification_report(test_gbt),
         "recall": round(recall, 3), "specificity": round(specificity, 3), "precision": round(precision, 3),
         "balanced_accuracy": round((recall + specificity) / 2, 4), "brier_rescaled": round(brier, 4)},
        "B", model_dir=TIP_MODEL_DIR, tags=TAGS_TIP, extra_tags={"stage": "tuned"})

The three tuning searches of this project are logged as nested runs (one parent per search, one child per setting)

In [0]:
def rows_from_table(tbl, metric_col="cv_areaUnderROC", std_col="cv_std"):
    """Turn a CV results table (pandas or Spark) into the rows log_search expects.
    Every column that does not start with 'cv_' is treated as a hyperparameter."""
    pdf = tbl.toPandas() if hasattr(tbl, "toPandas") else tbl
    param_cols = [c for c in pdf.columns if not c.startswith("cv_")]
    rows = []
    for rec in pdf.to_dict("records"):
        params = {}
        for c in param_cols:
            v = rec[c]
            v = v.item() if hasattr(v, "item") else v          # numpy -> plain Python
            if isinstance(v, float) and v.is_integer() and c in ("maxDepth", "maxIter"):
                v = int(v)                                      # 6.0 -> 6
            params[c] = v
        std = rec.get(std_col)
        rows.append({
            "params": params,
            "metric": float(rec[metric_col]),
            "std": None if std is None or std != std else float(std),   # NaN -> None
        })
    return rows


searches = [
    ("tip_lr_search",         results_lr,   "LogisticRegression"),
    ("tip_gbt_search_round1", results_gbt,  "GBTClassifier"),
    ("tip_gbt_search_round2", results_gbt2, "GBTClassifier"),
]
for name, table, model in searches:
    n = log_search(name, "tip", rows_from_table(table),
                   metric_name="cv_areaUnderROC", higher_is_better=True,
                   fixed_params={"model": model})
    print(f"{name}: {n} child runs")

tip_lr_search: 6 child runs
tip_gbt_search_round1: 4 child runs
tip_gbt_search_round2: 4 child runs


In [0]:
runs_tip = mlflow.search_runs(filter_string="tags.task = 'tip'", order_by=["metrics.auc DESC"])
print(runs_tip[["tags.mlflow.runName", "metrics.auc", "metrics.auc_pr", "metrics.accuracy", "metrics.f1"]]
      .to_string(index=False))

              tags.mlflow.runName  metrics.auc  metrics.auc_pr  metrics.accuracy  metrics.f1
                        gbt_tuned        0.670           0.809             0.666       0.674
        logistic_regression_tuned        0.666           0.805             0.687       0.687
logistic_regression_class_weights        0.666           0.805             0.688       0.688
              logistic_regression        0.665           0.804             0.766       0.713
                    random_forest        0.651           0.796             0.759       0.689
          baseline_majority_class        0.500           0.716             0.716         NaN


### What was saved

- **Model:** `/Volumes/taxi/taxi_ml/ml_models/tip_gbt_tuned` holds the tuned GBT with its preprocessing (`PipelineModel`), `decision_settings.json` (threshold 0.50 on the raw score, positive class weight and the rescaling formula) and `manifest.json`. The reload was checked on 500 May trips and on the full May set.
- **Runs:** six MLflow runs with `task=tip` (majority-class baseline, three hand-set models, two tuned models), so they never mix with the fare runs. Every MLflow run also logs the data versions as parameters and lists its input tables as dataset inputs.
- **Manifest:** `manifest.json` in the model folder records the Delta versions and row counts of the data used, the features with their types, the split, the settings, the test metrics and the software versions. Every MLflow run also logs the data versions as parameters and lists its input tables as dataset inputs.
- **To score new trips:** load the model, take the probability of class 1, flag a trip when it is at least 0.50, and apply the rescaling formula to read it as a probability.

### Conclusions for Project B

* **The untuned models are close to "always predict a high tip".** The random forest finds 99.7% of the high tips but only 15.7% of the low tips on May, no threshold gives a clearly useful operating point, and class weights change where the threshold sits, not the ranking (AUC 0.666 against 0.665).
* **The signal is real but modest.** The tuned GBT reaches AUC-ROC 0.670 on the untouched May data (0.5 is no skill), and AUC-PR 0.809 against 0.716 for always predicting a high tip. A plausible reason, not tested here: whether a rider tips 20% or more depends mostly on the rider, which the trip data does not contain.
* **Tuning mattered less than the process.** Depth-5 trees overfit (CV AUC 0.660 to 0.664), while depth 2 and 3 are tied within about 0.005 of each other (0.671 to 0.675); the one-standard-error rule chose the simplest, `maxDepth=2`, `maxIter=40`. The additive logistic regression (0.666) is close behind, so interactions between features add little.
* **The class mix drove three choices.** High tips are the majority (71.7%, stable at 0.715 to 0.719 across months), so `areaUnderROC` was the tuning metric (a no-skill AUC-PR is already 0.72), and balanced accuracy chose the threshold, because F1 for the positive class would have picked a threshold flagging over 93% of trips.
* **The scores need rescaling to be read as probabilities.** Class weights make the raw scores run 0.2 below the real rates; rescaling with `p / (p + w_pos * (1 - p))` brings every decile within 0.021 on April, and the mean prediction on May is 0.717 against an actual 0.716 (Brier 0.173 against 0.203 for the base rate).
* **A random split is slightly optimistic for the tuned model.** It scores about 0.006 AUC higher than the time-based split (for the hand-set models the two agreed within 0.003), so the time-based number is the one to report.

## Project C: grouping pickup zones (clustering with KMeans)

**Goal:** find groups of pickup zones that behave alike, without any label. Each zone becomes one row of averages and shares computed from the **full** Silver table (not the 2% sample). Zones with fewer than 500 trips are dropped because their averages would be noisy. The features are standardised, k is chosen with the elbow and silhouette plots, and the clusters are named by looking at the busiest zones in each.

### Step 1: one row per zone

| Feature | Meaning |
|---|---|
| `log_trips` | log10 of the number of pickups (counts are very skewed) |
| `avg_distance`, `avg_fare`, `avg_duration` | average trip distance, fare and duration |
| `night_share` | share of trips starting between 22:00 and 04:59 |
| `weekend_share` | share of trips on Saturday or Sunday |
| `airport_share` | share of airport trips |

In [0]:
MIN_TRIPS = 500     # zones with fewer pickups have noisy averages and are dropped

zone_profile_df = (
    spark.table("taxi.taxi_silver.yellow_taxi_trips_cleaned")        # full Silver table, not the sample
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("trips"),
        F.avg("trip_distance").alias("avg_distance"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_duration_mins").alias("avg_duration"),
        F.avg(F.when((F.col("pickup_hour") >= 22) | (F.col("pickup_hour") < 5), 1.0).otherwise(0.0)).alias("night_share"),
        F.avg(F.when(F.col("pickup_day_of_week").isin(1, 7), 1.0).otherwise(0.0)).alias("weekend_share"),
        F.avg(F.when(F.col("trip_type") == "airport", 1.0).otherwise(0.0)).alias("airport_share"),
    )
    .filter(F.col("trips") >= MIN_TRIPS)
    # Pickup counts are very skewed (a few zones have far more trips than most): use the log
    .withColumn("log_trips", F.log10("trips"))
)

zone_profile_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("taxi.taxi_ml.zone_profile")

zone_profile = spark.table("taxi.taxi_ml.zone_profile")
print(f"zones: {zone_profile.count()}")

zones: 216


### Step 2: scale, then choose k

KMeans groups zones by Euclidean distance, so a feature measured in large units (a fare in dollars) would swamp one measured as a share (0 to 1). All features are therefore standardised to mean 0 and standard deviation 1 first.

k is chosen with two diagnostics, computed for k = 2 to 10:

- **Elbow (within-cluster cost):** the cost always falls as k grows; look for the k after which it stops falling quickly.
- **Silhouette score:** how well each zone fits its own cluster compared with the nearest other one (1 = well separated, 0 = overlapping). Higher is better.

In [0]:
feature_cols = ["log_trips", "avg_distance", "avg_fare", "avg_duration",
                "night_share", "weekend_share", "airport_share"]

prep = Pipeline(stages=[
    VectorAssembler(inputCols=feature_cols, outputCol="raw"),
    StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True),
]).fit(zone_profile)

zones_vec = prep.transform(zone_profile)

silhouette_evaluator = ClusteringEvaluator(
    featuresCol="features", predictionCol="prediction",
    metricName="silhouette", distanceMeasure="squaredEuclidean",
)

k_results = []      # one (k, cost, silhouette) tuple per k; created before the loop so it keeps every k
for k in range(2, 11):
    model_k = KMeans(k=k, seed=42, featuresCol="features", maxIter=50).fit(zones_vec)
    cost = model_k.summary.trainingCost
    silhouette = silhouette_evaluator.evaluate(model_k.transform(zones_vec))
    k_results.append((k, cost, silhouette))
    print(f"k={k:2d} cost={cost:.2f} silhouette={silhouette:.3f}")

k= 2 cost=826.89 silhouette=0.653
k= 3 cost=606.52 silhouette=0.703
k= 4 cost=507.96 silhouette=0.536
k= 5 cost=352.69 silhouette=0.517
k= 6 cost=299.99 silhouette=0.552
k= 7 cost=255.32 silhouette=0.499
k= 8 cost=257.42 silhouette=0.432
k= 9 cost=238.97 silhouette=0.447
k=10 cost=225.18 silhouette=0.449


In [0]:
k_values = [r[0] for r in k_results]
costs = [r[1] for r in k_results]
silhouettes = [r[2] for r in k_results]

fig, (ax_elbow, ax_silhouette) = plt.subplots(1, 2, figsize=(14, 5))

ax_elbow.plot(k_values, costs, marker="o")
ax_elbow.set_title("Elbow method: within-cluster cost")
ax_elbow.set_xlabel("Number of clusters (k)")
ax_elbow.set_ylabel("Within-cluster cost")

ax_silhouette.plot(k_values, silhouettes, marker="o")
ax_silhouette.set_title("Silhouette score")
ax_silhouette.set_xlabel("Number of clusters (k)")
ax_silhouette.set_ylabel("Silhouette score (higher is better)")

for ax in (ax_elbow, ax_silhouette):
    ax.set_xticks(k_values)
    ax.grid(True)

plt.tight_layout()
plt.show()

### Choosing k

The silhouette score is highest at k = 3 (0.70), but that solution isolates only the three airport zones, which `trip_type` already identifies, and leaves the other 213 zones in two broad groups (a Manhattan group of 71 zones and a residential/outer group of 142). k = 6 has a lower silhouette (about 0.55) but is the next local peak, and all six of its clusters can be interpreted and add information the data does not already carry: a downtown nightlife cluster (night share 41.6%) separated from the Manhattan core, and a group of outlying zones with a high airport share. The airport cluster is identical at both values of k, which suggests it is stable.

The elbow plot also supports a larger k than 3: the cost falls steeply until k = 5 (353) and flattens from k = 7 (255). Between those, k = 6 is the better choice because its silhouette is higher (0.552 against 0.517 at k = 5 and 0.499 at k = 7).

**Caution:** three of the six clusters are small (3, 10 and 12 zones), so their averages rest on few zones. The seed check in Step 5 shows that the airport, nightlife and Manhattan core groups are robust, while the split of the remaining zones into residential, outer-borough residential and outlying airport-heavy groups changes with the random seed.

### Step 3: fit the final model and name the clusters

KMeans only returns numbers 0 to 5. The names come from two views: the average profile of each cluster, and its busiest zones (joined from the TLC zone lookup).

In [0]:
best_k = 6

kmeans_model = KMeans(k=best_k, seed=42, featuresCol="features", maxIter=50).fit(zones_vec)

lookup = spark.table("taxi.taxi_bronze.yellow_taxi_lookup_raw_data").select("LocationID", "Borough", "Zone")

clustered = (
    kmeans_model.transform(zones_vec)
    .join(lookup, zones_vec.PULocationID == lookup.LocationID)
    .drop("LocationID")
    .withColumnRenamed("prediction", "cluster")
)

# Save the assignment so other notebooks and dashboards can reuse it
(clustered.select("PULocationID", "Borough", "Zone", "cluster", *feature_cols)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("taxi.taxi_ml.zone_clusters"))

# What does each cluster look like on average?
(clustered.groupBy("cluster")
    .agg(F.count("*").alias("zones"),
         F.round(F.avg("trips"), 0).alias("avg_trips"),
         F.round(F.avg("avg_distance"), 2).alias("avg_distance"),
         F.round(F.avg("avg_fare"), 2).alias("avg_fare"),
         F.round(F.avg("night_share"), 3).alias("night_share"),
         F.round(F.avg("weekend_share"), 3).alias("weekend_share"),
         F.round(F.avg("airport_share"), 3).alias("airport_share"))
    .orderBy("cluster").show())

# Which zones are in each cluster? (the 8 busiest per cluster)
# collect_list after a groupBy does not keep an earlier orderBy, so sort inside the aggregation:
# sort_array on (trips, Zone) structs orders by trips first, then by Zone for ties.
(clustered.groupBy("cluster")
    .agg(
        F.transform(
            F.slice(F.sort_array(F.collect_list(F.struct("trips", "Zone")), asc=False), 1, 8),
            lambda x: x["Zone"]).alias("busiest_zones"))
    .orderBy("cluster")
    .show(truncate=False))

+-------+-----+---------+------------+--------+-----------+-------------+-------------+
|cluster|zones|avg_trips|avg_distance|avg_fare|night_share|weekend_share|airport_share|
+-------+-----+---------+------------+--------+-----------+-------------+-------------+
|      0|   58|   5578.0|        6.38|   30.17|      0.071|        0.201|        0.008|
|      1|   12|  89254.0|        3.77|   22.69|      0.416|        0.465|        0.014|
|      2|   48| 196745.0|        2.92|   18.08|      0.122|        0.256|        0.012|
|      3|   10|   3571.0|       10.21|   56.16|      0.117|        0.277|        0.137|
|      4|    3| 383297.0|       11.56|   50.67|      0.186|        0.265|        0.942|
|      5|   85|   2542.0|       10.42|    39.6|      0.046|        0.152|        0.006|
+-------+-----+---------+------------+--------+-----------+-------------+-------------+

+-------+--------------------------------------------------------------------------------------------------------------

### Cluster profiles (k = 6)

| Cluster | Name | Zones | Avg pickups per zone | Avg distance (miles) | Avg fare ($) | Night share | Weekend share | Airport share |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 0 | Residential neighbourhoods, medium volume (Upper Manhattan, Brooklyn, Queens) | 58 | 5,578 | 6.38 | 30.17 | 7.1% | 20.1% | 0.8% |
| 1 | Downtown nightlife (Williamsburg, North Side, is the one Brooklyn zone among its eight busiest) | 12 | 89,254 | 3.77 | 22.69 | 41.6% | 46.5% | 1.4% |
| 2 | Manhattan core (Midtown and the Upper East Side) | 48 | 196,745 | 2.92 | 18.08 | 12.2% | 25.6% | 1.2% |
| 3 | Outlying zones with many airport trips | 10 | 3,571 | 10.21 | 56.16 | 11.7% | 27.7% | 13.7% |
| 4 | Airports | 3 | 383,297 | 11.56 | 50.67 | 18.6% | 26.5% | 94.2% |
| 5 | Outer-borough residential, very low volume | 85 | 2,542 | 10.42 | 39.60 | 4.6% | 15.2% | 0.6% |

### Reading the clusters (k = 6)

- **Cluster 0, residential neighbourhoods (58 zones):** Central Harlem North, Washington Heights South, Crown Heights North, East New York, Sunnyside. Medium volume (about 5,600 pickups per zone) and longer trips (6.4 miles, $30.17), with a low night share (7.1%).
- **Cluster 1, downtown nightlife (12 zones):** East Village, West Village, Lower East Side, Greenwich Village South and Meatpacking. Its **night share is 41.6% and its weekend share 46.5%**, against at most 18.6% (the airports) and 27.7% in any other cluster. At k = 3 these zones were hidden inside the Manhattan core.
- **Cluster 2, Manhattan core (48 zones):** the highest volume among regular zones (about 197,000 pickups per zone), the shortest trips (2.9 miles) and the lowest fare ($18.08). It contains Upper East Side South and North, Midtown Center and East, Penn Station and Times Sq/Theatre District.
- **Cluster 3, outlying zones with many airport trips (10 zones):** South Ozone Park, Baisley Park, Springfield Gardens South, Kew Gardens, Long Island City/Hunters Point, Red Hook and the "Outside of NYC" zone. Their airport share (13.7%) is at least about ten times that of the other non-airport clusters, and their average fare ($56.16) is the highest of all six clusters, the airports included. South Ozone Park is also the most profitable corridor in the Gold tables.
- **Cluster 4, airports (3 zones):** JFK, LaGuardia and East Elmhurst, with 94.2% airport trips. This cluster is the same at k = 3.
- **Cluster 5, outer-borough residential, very low volume (85 zones):** Canarsie, Flatlands, Coney Island, Jamaica, Co-Op City, Bay Ridge, Starrett City and Soundview/Castle Hill, which are spread across outer Brooklyn, Queens and the Bronx. This cluster has the lowest volume (about 2,500 pickups per zone), the lowest night and weekend shares (4.6% and 15.2%), and long trips (10.4 miles, $39.60). Long trips from these zones are plausible, since riders head to Manhattan or the airports, but that is not tested here.

**Caveat:** clusters 1, 3 and 4 contain only 12, 10 and 3 zones, so their profiles rest on few zones. Clusters 0, 3 and 5 are the soft part: Step 5 shows that their boundaries move with the random seed. The names come from reading the profile and the busiest zones. KMeans only supplies the grouping.

### Step 4: what k = 3 hid

In [0]:
# Compare the k = 3 solution with the final k = 6 solution:
# rows = k=3 clusters, columns = k=6 clusters, values = number of zones
model_k3 = KMeans(k=3, seed=42, featuresCol="features", maxIter=50).fit(zones_vec)
k3_labels = model_k3.transform(zones_vec).select("PULocationID", F.col("prediction").alias("cluster_k3"))

(clustered.join(k3_labels, on="PULocationID")
    .groupBy("cluster_k3").pivot("cluster").count()
    .na.fill(0)
    .orderBy("cluster_k3").show())

+----------+---+---+---+---+---+---+
|cluster_k3|  0|  1|  2|  3|  4|  5|
+----------+---+---+---+---+---+---+
|         0| 11| 12| 48|  0|  0|  0|
|         1| 47|  0|  0| 10|  0| 85|
|         2|  0|  0|  0|  0|  3|  0|
+----------+---+---+---+---+---+---+



**Reading the k = 3 versus k = 6 comparison.** Each row is a k = 3 cluster and each column a k = 6 cluster. The three airport zones are alone in one k = 3 cluster and form k = 6 cluster 4, so the airport cluster is identical at both values of k. The k = 3 group of 71 zones combines the Manhattan core (48), the downtown nightlife zones (12) and 11 zones that k = 6 calls residential, so k = 3 hides the nightlife zones inside it. The k = 3 group of 142 zones combines the outer-borough residential zones (85), the residential neighbourhoods (47) and the 10 outlying airport-heavy zones, which k = 3 also hides.

### Step 5: is the grouping stable?

KMeans starts from random centres, so another random seed can give different groups. The final model (seed 42) is compared with ten refits using other seeds. For each run, the **adjusted Rand index** measures how similar its groups are to the final ones: 1 means identical groups (the cluster numbers may differ, only which zones sit together counts) and 0 means no more agreement than chance. The table also shows each run's cost and silhouette, so a worse local solution can be recognised, and the sizes of its six clusters.

In [0]:
# Reference: the final k = 6 assignment (seed 42)
reference = clustered.select("PULocationID", "Zone", F.col("cluster").alias("ref")).toPandas()


def adjusted_rand_index(a, b):
    """Adjusted Rand index of two labelings of the same zones: 1 = identical groups, about 0 = chance level."""
    table = pd.crosstab(a, b).to_numpy()
    pairs = lambda x: x * (x - 1) / 2          # number of pairs inside each count
    sum_cells = pairs(table).sum()
    sum_rows = pairs(table.sum(axis=1)).sum()
    sum_cols = pairs(table.sum(axis=0)).sum()
    expected = sum_rows * sum_cols / pairs(table.sum())
    return (sum_cells - expected) / ((sum_rows + sum_cols) / 2 - expected)


labelings, rows = {}, []
for seed in [42, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:         # 42 is the final model: it must reproduce itself
    model_s = KMeans(k=6, seed=seed, featuresCol="features", maxIter=50).fit(zones_vec)
    pred_s = model_s.transform(zones_vec)
    labels = (pred_s.select("PULocationID", F.col("prediction").alias("label")).toPandas()
              .merge(reference, on="PULocationID"))
    labelings[seed] = labels
    rows.append({
        "seed": seed,
        "cost": round(model_s.summary.trainingCost, 2),
        "silhouette": round(silhouette_evaluator.evaluate(pred_s), 3),
        "adjusted_rand_vs_seed42": round(adjusted_rand_index(labels["label"], labels["ref"]), 3),
        "cluster_sizes": sorted(labels["label"].value_counts().tolist(), reverse=True),
    })

stability = pd.DataFrame(rows)
print(stability.to_string(index=False))

# Which zones change group in the least similar run?
worst_seed = int(stability.sort_values("adjusted_rand_vs_seed42").iloc[0]["seed"])
lab = labelings[worst_seed].copy()
print(f"\nLeast similar run: seed {worst_seed} (rows = final clusters, columns = clusters of that run)")
print(pd.crosstab(lab["ref"], lab["label"]).to_string())

# A zone 'moves' if its final cluster is not the one that most of its new cluster's zones came from
majority_ref = lab.groupby("label")["ref"].agg(lambda s: s.value_counts().idxmax())
lab["expected_ref"] = lab["label"].map(majority_ref)
moved = lab[lab["ref"] != lab["expected_ref"]].sort_values(["ref", "Zone"])
print(f"\nZones that change group in seed {worst_seed}: {len(moved)} of {len(lab)}")
if len(moved):
    print(moved[["Zone", "ref", "label"]].rename(columns={"ref": "final_cluster", "label": f"seed_{worst_seed}_cluster"})
          .to_string(index=False))

 seed   cost  silhouette  adjusted_rand_vs_seed42           cluster_sizes
   42 299.99       0.552                    1.000 [85, 58, 48, 12, 10, 3]
    1 294.86       0.517                    0.527 [83, 47, 39, 33, 11, 3]
    2 296.07       0.504                    0.448 [83, 43, 41, 35, 11, 3]
    3 300.35       0.543                    0.798 [77, 63, 50, 12, 11, 3]
    4 299.44       0.541                    0.810 [75, 63, 49, 14, 12, 3]
    5 300.35       0.543                    0.798 [77, 63, 50, 12, 11, 3]
    6 296.08       0.500                    0.448 [82, 43, 42, 35, 11, 3]
    7 325.07       0.540                    0.465   [95, 54, 50, 8, 6, 3]
    8 304.32       0.458                    0.523 [69, 66, 41, 26, 11, 3]
    9 304.32       0.458                    0.523 [69, 66, 41, 26, 11, 3]
   10 298.40       0.540                    0.662 [69, 65, 55, 12, 12, 3]

Least similar run: seed 2 (rows = final clusters, columns = clusters of that run)
label   0   1   2   3  4   5


**Reading the stability check.** Seed 42 reproduces itself (adjusted Rand index 1.000). The ten other seeds give an index between 0.448 and 0.810, so the six groups as a whole are not reproduced exactly: only three runs reach about 0.8 (seeds 3, 4 and 5; seeds 3 and 5 found the same solution). The least similar run (seed 2) shows what is stable and what is not.

- **Stable:** the three airport zones are together in every run (a cluster of 3 appears in all eleven), 11 of the 12 downtown nightlife zones stay together, and 43 of the 48 Manhattan core zones stay together.
- **Not stable:** the split of the other 153 zones. In seed 2 the 85 outer-borough residential zones are cut into two groups (50 and 35 zones), the 58 residential neighbourhoods into two (32 and 26), and the 10 outlying airport-heavy zones are dissolved into three other clusters. A plausible reason, not tested here: these groups differ in volume and trip length, which change gradually from zone to zone, so where KMeans draws the lines depends on its starting point.
- **Which solution is reported:** the final model (seed 42) has the highest silhouette of the eleven runs (0.552 against at most 0.543) but not the lowest cost: five of the ten other runs reach a lower cost, by at most 1.7%.
- **Consequence:** the conclusions below treat the airports, the nightlife zones and the Manhattan core as firm groups, and the other three as one soft split.

### Step 6: save the model and track the runs

Project C has no test set, so there is no "score on unseen data". What is worth keeping is (1) the fitted pipeline (scaler and KMeans together, so raw zone features go in and a cluster comes out), (2) the cluster names, which so far exist only in markdown, and (3) the runs that led to k = 6. The reload check asks a different question than in Projects A and B: does the reloaded model put **every zone in the same cluster** as the model in memory?

In [0]:
CLUSTER_MODEL_DIR = "/Volumes/taxi/taxi_ml/ml_models/zone_kmeans_k6"

# One pipeline from raw zone features to a cluster (same settings as Step 3, so the same clusters)
cluster_pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=feature_cols, outputCol="raw"),
    StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True),
    KMeans(k=best_k, seed=42, featuresCol="features", maxIter=50),
]).fit(zone_profile)

cluster_pipeline.write().overwrite().save(CLUSTER_MODEL_DIR)
reloaded_c = PipelineModel.load(CLUSTER_MODEL_DIR)

final_labels = clustered.select("PULocationID", "cluster")

def zones_that_differ(model):
    """Number of zones whose cluster from this model differs from the final assignment of Step 3."""
    labels = model.transform(zone_profile).select("PULocationID", F.col("prediction").alias("other"))
    return final_labels.join(labels, "PULocationID").filter(F.col("cluster") != F.col("other")).count()

size_mb = sum(os.path.getsize(os.path.join(root, f))
              for root, _, files in os.walk(CLUSTER_MODEL_DIR) for f in files) / 1e6

print(f"zones: {zone_profile.count()} | different from Step 3, refitted pipeline: {zones_that_differ(cluster_pipeline)}"
      f" | different from Step 3, reloaded pipeline: {zones_that_differ(reloaded_c)} | model folder: {size_mb:.2f} MB")

zones: 216 | different from Step 3, refitted pipeline: 0 | different from Step 3, reloaded pipeline: 0 | model folder: 0.02 MB


In [0]:
cost_k, silhouette_k = [(c, s) for k, c, s in k_results if k == best_k][0]
others = stability[stability["seed"] != 42]["adjusted_rand_vs_seed42"]

manifest_c = write_manifest(
    CLUSTER_MODEL_DIR, "zone_kmeans_k6",
    tables=["taxi.taxi_ml.zone_profile"],
    feature_df=zone_profile, categorical=[], numeric=feature_cols,
    split={"method": "none (unsupervised)", "zones": zone_profile.count(), "min_trips_per_zone": MIN_TRIPS},
    hyperparameters={"model": "KMeans", "k": best_k, "seed": 42, "maxIter": 50,
                     "scaling": "StandardScaler (mean 0, std 1), fitted on the zone profile"},
    metrics={"silhouette": round(silhouette_k, 3), "cost": round(cost_k, 2)},
    extra={"seed_stability_ari": {"min": round(float(others.min()), 3), "mean": round(float(others.mean()), 3),
                                  "max": round(float(others.max()), 3)},
           "assignment_table": "taxi.taxi_ml.zone_clusters"},
)
print(json.dumps({k: manifest_c[k] for k in ["input_tables", "metrics", "seed_stability_ari"]}, indent=2))

{
  "input_tables": [
    {
      "table": "taxi.taxi_ml.zone_profile",
      "delta_version": 21,
      "last_write": "2026-11-28 09:27:48",
      "rows": 216
    }
  ],
  "metrics": {
    "silhouette": 0.552,
    "cost": 299.99
  },
  "seed_stability_ari": {
    "min": 0.448,
    "mean": 0.6,
    "max": 0.81
  }
}


In [0]:
# Guard: the names below belong to the cluster numbers and sizes of the seed-42 solution
expected_sizes = {0: 58, 1: 12, 2: 48, 3: 10, 4: 3, 5: 85}
actual_sizes = {r["cluster"]: r["n"] for r in clustered.groupBy("cluster").agg(F.count("*").alias("n")).collect()}
assert actual_sizes == expected_sizes, f"cluster numbers or sizes changed: {actual_sizes}"

cluster_names = spark.createDataFrame([
    (0, "Residential neighbourhoods, medium volume", "soft"),
    (1, "Downtown nightlife", "robust"),
    (2, "Manhattan core", "robust"),
    (3, "Outlying zones with many airport trips", "soft"),
    (4, "Airports", "robust"),
    (5, "Outer-borough residential, very low volume", "soft"),
], ["cluster", "name", "stability_across_seeds"])

(cluster_names.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("taxi.taxi_ml.zone_cluster_names"))

# Join with the assignment: what a dashboard or another notebook would query
(spark.table("taxi.taxi_ml.zone_clusters")
    .join(spark.table("taxi.taxi_ml.zone_cluster_names"), "cluster")
    .select("PULocationID", "Zone", "cluster", "name", "stability_across_seeds")
    .orderBy("cluster", "Zone").show(8, truncate=False))

+------------+----------------+-------+-----------------------------------------+----------------------+
|PULocationID|Zone            |cluster|name                                     |stability_across_seeds|
+------------+----------------+-------+-----------------------------------------+----------------------+
|7           |Astoria         |0      |Residential neighbourhoods, medium volume|soft                  |
|12          |Battery Park    |0      |Residential neighbourhoods, medium volume|soft                  |
|17          |Bedford         |0      |Residential neighbourhoods, medium volume|soft                  |
|25          |Boerum Hill     |0      |Residential neighbourhoods, medium volume|soft                  |
|26          |Borough Park    |0      |Residential neighbourhoods, medium volume|soft                  |
|33          |Brooklyn Heights|0      |Residential neighbourhoods, medium volume|soft                  |
|35          |Brownsville     |0      |Residential neig

In [0]:
TAGS_CLUSTER = {"task": "cluster", "split": "none (unsupervised)", "seed": "42",
                "sample": f"full Silver table, zones with at least {MIN_TRIPS} pickups"}

n_zones = zone_profile.count()
base_params = {"model": "KMeans", "seed": 42, "maxIter": 50, "n_zones": n_zones,
               "scaling": "StandardScaler (mean 0, std 1)", "features": ",".join(feature_cols)}

# The elbow / silhouette sweep from Step 2: one run per k (the final k gets its own run below)
for k, cost, silhouette in k_results:
    if k != best_k:
        log_run(f"kmeans_k{k}", {**base_params, "k": k},
                {"cost": round(cost, 2), "silhouette": round(silhouette, 3)},
                "zone profile", tags=TAGS_CLUSTER, extra_tags={"stage": "sweep"})

# The final model: scores, cluster sizes, stability over the ten other seeds, and the saved pipeline
final_cost, final_sil = [(c, s) for k, c, s in k_results if k == best_k][0]
others = stability[stability["seed"] != 42]["adjusted_rand_vs_seed42"]
log_run(f"kmeans_k{best_k}", {**base_params, "k": best_k},
        {"cost": round(final_cost, 2), "silhouette": round(final_sil, 3),
         "smallest_cluster": int(min(expected_sizes.values())),
         "ari_min": round(float(others.min()), 3), "ari_mean": round(float(others.mean()), 3),
         "ari_max": round(float(others.max()), 3)},
        "zone profile", model_dir=CLUSTER_MODEL_DIR, tags=TAGS_CLUSTER,
        extra_tags={"stage": "final", "assignment_table": "taxi.taxi_ml.zone_clusters"},
        figures={"elbow_silhouette.png": fig})

In [0]:
runs_c = mlflow.search_runs(filter_string="tags.task = 'cluster'", order_by=["metrics.silhouette DESC"])
print(runs_c[["tags.mlflow.runName", "params.k", "metrics.cost", "metrics.silhouette", "metrics.ari_mean"]]
      .to_string(index=False))

In [0]:
all_runs = mlflow.search_runs(order_by=["tags.task ASC"])
model_runs = all_runs[all_runs["tags.search_task"].isna()]            # finished models
search_runs = all_runs[all_runs["tags.search_task"].notna()]          # tuning parents + children
version_cols = [c for c in model_runs.columns if c.startswith("params.") and c.endswith("_delta_version")]

print(len(model_runs), "model runs | every model run has a data version:",
      bool(model_runs["params.silver_delta_version"].notna().all()))
print(len(search_runs), "tuning runs:",
      int((search_runs["tags.kind"] == "search").sum()), "searches,",
      int((search_runs["tags.kind"] == "trial").sum()), "settings")
print(model_runs[["tags.task", "tags.mlflow.runName"] + version_cols].head(12).to_string(index=False))

### What was saved

- **Model:** `/Volumes/taxi/taxi_ml/ml_models/zone_kmeans_k6` holds the scaler and KMeans as one `PipelineModel`: raw zone features in, cluster out. The reload gave the same cluster for all 216 zones.
- **Tables:** `taxi.taxi_ml.zone_clusters` (the assignment) and `taxi.taxi_ml.zone_cluster_names` (the six names, marked robust or soft from the seed check).
- **Runs:** nine MLflow runs with `task=cluster`: one per k from 2 to 10, and the final k = 6 run with the saved model and the stability scores (adjusted Rand index 0.45 to 0.81 over ten other seeds).
- **Manifest:** `manifest.json` in the model folder records the Delta versions and row counts of the data used, the features with their types, the split, the settings, the test metrics and the software versions. Every MLflow run also logs the data versions as parameters.

### Conclusions for Project C

- **Pickup zones fall into groups that differ in volume, trip length and when trips happen.** Three are robust across random seeds: the airports (3 zones), downtown nightlife (12) and the Manhattan core (48). The other 153 zones form a soft split into residential neighbourhoods (58), outer-borough residential (85) and outlying zones with many airport trips (10), whose boundaries move with the seed. The assignment is saved in `taxi.taxi_ml.zone_clusters` (names in `taxi.taxi_ml.zone_cluster_names`) and the fitted pipeline in `ml_models/zone_kmeans_k6`.
- **k was chosen by what it reveals, not by the silhouette alone.** k = 3 has the highest silhouette (0.70) but only isolates the three airport zones, which `trip_type` already identifies. k = 6 (0.55) also separates the nightlife zones (night share 41.6%) from the Manhattan core and a group of outlying zones with a high airport share (13.7%).
- **In the final grouping, trips get longer and more expensive away from Midtown.** Among the non-airport clusters, the outer-borough residential and outlying airport-heavy zones have the longest trips (10.4 and 10.2 miles) and the highest fares ($39.60 and $56.16), against 2.9 miles and $18.08 in the Manhattan core. These two groups are part of the soft split (Step 5), so the numbers describe the final grouping, not a fixed property of the zones.
- **Limits.** Three clusters have only 3, 10 and 12 zones, and the names are interpretations of the profiles and busiest zones. The seed check (adjusted Rand index 0.45 to 0.81 against the final model) shows that the split among the residential, outer-borough and outlying zones is not stable.
- **Possible use, not tested here:** the cluster label could replace the roughly 255 raw zone values as a compact feature in the fare and tip models.


## Summary

| Project | Best result | Takeaway |
|---|---|---|
| A: fare regression | At drop-off, gradient-boosted trees: RMSE $5.37, MAE $1.84, R² 0.914. At pickup, tuned gradient-boosted trees with native zone categories: RMSE $7.86, MAE $4.11, R² 0.816 (average baseline $18.32, zone-pair lookup $8.49) | Distance and duration explain most of the fare. At pickup the zone encoding mattered more than the tuning: a plain zone-pair lookup beat every one-hot model, and native categories brought the trees past it. A zone-pair feature lowers MAE to $3.76 but leaves RMSE unchanged |
| B: high-tip classification | Tuned gradient-boosted trees (time split, May test): AUC-ROC 0.670, AUC-PR 0.809 against 0.716 for always predicting a high tip; tuned logistic regression 0.666 / 0.805 | The features carry only a modest signal, and the tuned gradient-boosted trees are about 0.004 AUC above the tuned logistic regression, within one fold-to-fold spread, so the two are effectively tied. Class weights shift the raw scores, so they are rescaled before being read as probabilities. A random split gave the same AUC for the hand-set models (within 0.003) but is slightly optimistic for the tuned GBT (+0.004 to +0.007) |
| C: zone clustering | k = 6: Manhattan core (48 zones), downtown nightlife (12), residential (58), outer-borough residential (85), outlying airport-heavy zones (10), airports (3) | k = 6 gives a lower silhouette than k = 3 (about 0.55 against 0.70), but it separates the nightlife zones from the Manhattan core and adds a group of outlying airport-heavy zones that k = 3 hides. The airport, nightlife and Manhattan core groups are stable across random seeds; the split of the other zones, including that airport-heavy group, is not (adjusted Rand index 0.45 to 0.81) |

**Saved artefacts.** Four fitted pipelines are stored in the `ml_models` volume (the tuned fare GBT, the fare GBT with the zone-pair feature, the tuned tip GBT with its decision settings, and the zone KMeans pipeline), each reloaded and checked against the in-memory model. Twenty-five model runs (ten for fare, six for tips, nine for clustering) are tagged by `task`, so each project's runs can be compared on their own, and 37 tuning runs (five searches with 32 settings in all) are stored as nested runs under a separate `search_task` tag. Each saved model carries a manifest with the data versions, features, split, settings and software versions, and every model run logs the Delta versions of its data and lists its input tables as MLflow dataset inputs.

**Limitations**

- Models are trained on a 2% sample (about 245,000 trips). Hyperparameters are tuned for Project A (grid search with 3-fold cross-validation) and for the Project B classifier (4 month folds, searched on a 50% sample); the first-part Project B models use hand-set values, and Project C's k comes from the elbow, the silhouette and interpretability.
- The data covers five months, so seasonality and long-term drift cannot be assessed.
- Project A: the pickup models assume the destination zone is known. With one-hot zones they cannot represent route-specific fares (a zone-pair lookup beat all of them); native categories for the trees fix most of this.
- Project A: the zone-pair feature is built from out-of-fold training averages; its cross-validated gain ($0.22 RMSE) was slightly optimistic and the test set showed none, so the test score is the reference.
- Project A: the largest fare errors are long trips ending in zone 265, whose fares do not follow the meter, and 41 zero-fare trips (0.02% of the sample) are kept.
- Project B: the decision threshold was chosen on April, which was also one of the cross-validation months; May was used once. The raw scores need rescaling to be read as probabilities.
- Project C: the airport, nightlife and Manhattan core clusters are stable across ten other random seeds, but the split of the remaining 153 zones into three groups is not (adjusted Rand index 0.45 to 0.81 against the final model); the names come from reading the profiles.